# BLIP ITM-base — DIMER E2E image-text retrieval fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/blip-itm-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/blip-itm-pipeline/blob/main/tutorials/blip_itm_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Salesforce%2Fblip--itm--base--coco-ffcc4d?style=flat)](https://huggingface.co/Salesforce/blip-itm-base-coco) [![Upstream](https://img.shields.io/badge/Upstream-salesforce%2FBLIP-181717?style=flat&logo=github&logoColor=white)](https://github.com/salesforce/BLIP) [![arXiv](https://img.shields.io/badge/arXiv-2201.12086-b31b1b.svg)](https://arxiv.org/abs/2201.12086)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** image-text matching / retrieval and bounded supervised fine-tuning of the text encoder's last blocks, the ITC projections and the ITM head on a photograph/matching-captions dataset, using the pinned `Salesforce/blip-itm-base-coco` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/blip_itm_pipeline/`, at revision `e38e7a82b607`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `bed8ad38cb2d04a5a4bdf2d071b3c3c0a4aa724c` (~896 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `Salesforce/blip-itm-base-coco` snapshot (an 895 MB `pytorch_model.bin` pickle, opened only after its SHA-256 matches and with `weights_only=True`), reads the four text columns of one digest-pinned VizWiz-Captions shard from the Hugging Face Hub (about 0.5 MB over HTTP range requests, no credential) and the image column of its first two row groups (672 photographs, about 166 MB in two range reads, each photograph refused on any size or SHA-256 mismatch), cuts row group 0 by image into 208 / 40 / 70 training, validation and core-test photographs and appends row group 1's 321 captioned photographs to the test gallery (391 photographs, 1,737 captions), scores three drawn scenes through the inference contract with an input manifest and a rejection probe, measures the frozen model's retrieval over the gallery (recall@1/5/10 in both directions, ITM re-ranking, ITM pair accuracy) beside the chance and colour-keyword baselines, runs a bounded fine-tuning of the text encoder's last two blocks, the two ITC projections and the ITM head with BLIP's contrastive and matching objectives and validation-rsum epoch selection, scores the gallery again per category, re-scores the drawn scenes with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify score parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about forty minutes after the downloads; a CUDA runtime is used automatically when present and finishes in a few minutes.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip holding a `records.jsonl` (or `records.json`) of `{{id, image, captions}}` objects — `image` a file name inside the zip, `captions` one or more distinct matching captions (the first is the retrieval query), optional `category` — beside the image files. They pass through the same validation, seeded image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the VizWiz sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`Salesforce/blip-itm-base-coco` is the BLIP model of Li et al. (2022) fine-tuned for image-text retrieval on COCO — a ViT-B/16 image encoder at 384×384 and a 12-layer BERT-style text encoder whose blocks also cross-attend to the image; 223,744,258 parameters, published under the **BSD-3-Clause** licence. It gives two scores per image–caption pair: the **ITC cosine** between the projected image and text embeddings (a dual-encoder score that can rank a gallery cheaply) and the **ITM probability** from a classifier head over the fused encoding (a per-pair score that is more accurate and far more expensive). **Neither score is calibrated and neither abstains**: a caption set with nothing that fits an image still yields a highest-scoring caption.

What this notebook adds to inference is **adaptation with matching captions**. The dataset is real and out of the model's distribution: VizWiz-Captions (Gurari et al., ECCV 2020; **CC BY 4.0**) — photographs taken by blind people, each with up to five crowd-written captions that name what is held and what a label says — a population the model never saw. The notebook reads only the four text columns of one pinned Hub shard (about 0.5 MB over HTTPS range requests) and the image column of its **first two row groups** (672 photographs in two range reads, each pinned by size and SHA-256 in the carried module). Row group 0 is split by image into 208 / 40 / 70; row group 1's 321 captioned photographs join the **test gallery** so that retrieval is scored over 391 photographs and 1,737 captions — on a 70-photograph gallery the frozen model is already at the ceiling (the build record measured rsum 5.72 of 6), and a gallery that small says nothing about adaptation. The honest question is narrow: does a bounded fine-tuning of the text encoder's last blocks, the projections and the ITM head on 208 photographs move recall over a real gallery, in which direction, and does the ITM re-ranking move with it? The retrieval protocol is implemented in the carried modules (**recall@1/5/10** in both directions with every caption in the gallery, **median rank**, **rsum**, **ITM pair accuracy** against the hardest wrong caption), and two **non-neural baselines** — **chance** and the **colour-keyword nearest neighbour** — show where a system with no model sits. Nothing here is a quality claim about your photographs: it is one seeded split of one corpus.

**Weight-format note:** upstream hosts no SafeTensors at the pinned revision; the carried module executes the digest-pinned `pytorch_model.bin` (a pickle, deserialised with `weights_only=True` after its SHA-256 is checked), while the `tf_model.h5` upstream also hosts is DIMER's upload artifact and is never loaded here. Section 3 stages and digest-verifies the snapshot before the processor or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot (a pickle checkpoint, and why that matters); read the captions of a digest-pinned corpus without downloading its shard and its photographs from two pinned row groups with per-file digests, validate them and split by image without leakage; score drawn scenes through the public API and read the two scores correctly (uncalibrated, no abstention); measure the frozen model's retrieval over a real gallery with recall@k in both directions and ITM re-ranking beside two non-neural baselines and read the per-category breakdown; run a bounded fine-tuning with BLIP's two objectives, explicit hyperparameters and validation-based epoch selection; evaluate on an image-disjoint gallery; re-score drawings from a different image family with the adapted model; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** captioning or question answering (separate checkpoints), retrieval over thousands of candidates with a precomputed index (the gallery here is 391 photographs, embedded in the kernel), reading text in the image (BLIP is not an OCR model, even though VizWiz captions transcribe labels), non-English captions, batch throughput, evaluation on COCO or Flickr30k (not bundled), fine-tuning of the vision encoder or the text embeddings, training on images that are not the pinned sample or your own uploads, and any claim that a VizWiz split stands in for your photographs. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is slow but adequate: the build record measured about 236 s to embed and score the 391-photograph gallery with top-5 ITM re-ranking and 561 s for the four epochs of fine-tuning (888 photograph–caption pairs per epoch, three fused passes per pair for the matching loss), including the one-off encoding of the 208 training photographs and the per-epoch validation scoring; the whole default path took 1,512 s with the snapshot and photographs already cached, and about four minutes on an RTX 5070 Ti. The pinned `torch==2.14.0` install and the 895 MB checkpoint are the large downloads of the run; the two row groups of photographs add about 166 MB.
- **Knowledge:** basic Python and PIL; what a dual-encoder (contrastive) score and a fused-encoder (matching) score are and why one is cheap and the other accurate; why a pickle checkpoint needs a digest check before `torch.load`; what recall@k, median rank and rsum measure over a gallery and why a small gallery inflates them; why a high match probability is not a correct match.
- **Data contract:** records are `{{id, image, captions}}` — an image file decodable by Pillow with sides between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px and one or more distinct, non-empty matching captions of at most `MAX_TEXT_CHARS` (256) characters (`MIN_CAPTIONS` = 1; VizWiz supplies up to five; the first caption is the retrieval query); optional `image_id` (defaults to the id) groups records on the same image and optional `category` labels the breakdown (`text` / `no-text` in the sample, from the corpus's text-detected flag). Ids match `[A-Za-z0-9_.:-]{{1,64}}` and are unique; a dataset needs 8..5,000 records; every record on the same image lands in the same split so a test image is never trained on; every (image, caption) pair is one positive training sample and the batch supplies the negatives. BYOD accepts one zip of images plus a `records.jsonl` / `records.json` in that shape.
- **Validation is structural, not semantic:** every image is opened and decoded and every caption checked, but nothing checks that a caption matches its image — a mislabelled corpus is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — photographs of people, documents or homes are exactly that. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads three parts of one object in the Hub dataset repository `mm-eval/VizWiz-Captions` at the immutable revision `c4a6d897…` (`data/val-00004-of-00005.parquet`, 392,245,504 bytes, SHA-256 `4492465a…`): the declared size and SHA-256 are checked against the pins before any byte is read; the four text columns of all 1,550 rows are fetched over HTTPS range requests through `pyarrow` (only the parquet footer and those column chunks) and refused unless their decoded SHA-256 matches; then the image column of row groups 0 and 1 only (672 JPEG files, about 166 MB) is read the same way, each photograph pinned by size and SHA-256 in the carried module and refused on any mismatch. The corpus is CC BY 4.0 (Gurari et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `Salesforce/blip-itm-base-coco` snapshot (~896 MB in total) at revision `bed8ad38cb2d…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'blip-itm-pipeline',
    'repository_revision': 'e38e7a82b607ffc602d239ab5f3acedff3c02d3d',
    'embedded_module': 'src/blip_itm_pipeline/pipeline.py',
    'embedded_modules': ['src/blip_itm_pipeline/metrics.py', 'src/blip_itm_pipeline/pipeline.py', 'src/blip_itm_pipeline/samples.py'],
    'module_sha256': '0c1dde1f1dd720104914c1233d0780bb9f3ec8f9820ccdc0b125ed9557b09568',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/blip_itm_pipeline/` @ `e38e7a82b607`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/blip_itm_pipeline/metrics.py`

In [ ]:
"""Corpus-level retrieval metrics and two non-neural baselines, in pure Python / numpy.

`pipeline.py` keeps the per-grid plumbing check (`recall_at_1`); this module implements the retrieval
protocol a COCO- or Flickr-style result is read by, over a set of records with several matching captions
each:

- the **gallery** is every caption of every record (about 4.6 per photograph in the VizWiz sample), the
  queries are every photograph (image -> text) and every caption (text -> image);
- **recall@k** (k = 1, 5, 10) in each direction: image -> text counts a hit when *any* of the image's
  captions is among the top-k captions; text -> image counts a hit when the caption's own photograph is
  among the top-k images; `median_rank` is the median rank of the first correct candidate; `rsum` is the
  sum of the six recalls (the number retrieval papers select on);
- **ITM pair accuracy**: for each photograph, its query caption against the hardest wrong caption (the
  highest ITC cosine among captions of other photographs); the fraction of photographs where the ITM
  head's match probability ranks the true pair first (chance 0.5).

Two baselines a fine-tuned model must beat: **chance** (the analytical expectation of a random ranking
over the same gallery) and **colour-keyword nearest neighbour** (text -> image: the query's closest
training caption by bag-of-words F1 names a training photograph whose 3x3 mean-colour grid ranks the
candidate photographs; image -> text: the photograph's colour-nearest training photograph lends its
captions, which rank the candidate captions by bag-of-words F1 — a lookup that knows the image through
27 numbers and the text through word overlap).
"""

from __future__ import annotations

import math
import re
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

RECALL_KS = (1, 5, 10)
COLOUR_GRID = 3
_PUNCT_RE = re.compile(r"[^\w\s]")
METRIC_DEFINITIONS = {
    "i2t_recall_at_k": (
        "fraction of photographs with at least one of their own captions among the top-k captions of the "
        "gallery under the score; in 0..1"
    ),
    "t2i_recall_at_k": "fraction of captions whose own photograph is among the top-k photographs; in 0..1",
    "median_rank": "median over queries of the rank (1-based) of the first correct candidate",
    "rsum": "i2t R@1 + R@5 + R@10 + t2i R@1 + R@5 + R@10; in 0..6",
    "itm_pair_accuracy": (
        "fraction of photographs whose query caption receives a higher ITM match probability than the "
        "hardest wrong caption (highest ITC cosine among other photographs' captions); chance 0.5"
    ),
}


def caption_tokens(text: str) -> list[str]:
    return _PUNCT_RE.sub(" ", text.lower()).split()


def unigram_f1(prediction: str, reference: str) -> float:
    """Bag-of-words F1 between two captions (multiset overlap)."""
    pred, ref = caption_tokens(prediction), caption_tokens(reference)
    if not pred or not ref:
        return 0.0
    counts: dict[str, int] = {}
    for token in ref:
        counts[token] = counts.get(token, 0) + 1
    overlap = 0
    for token in pred:
        if counts.get(token, 0) > 0:
            overlap += 1
            counts[token] -= 1
    if not overlap:
        return 0.0
    precision, recall = overlap / len(pred), overlap / len(ref)
    return 2 * precision * recall / (precision + recall)


def gallery(records: Sequence[Mapping[str, Any]]) -> tuple[list[str], list[int]]:
    """Every caption of every record in record order, with the owning record index of each caption."""
    texts, owners = [], []
    for index, record in enumerate(records):
        for caption in record["captions"]:
            texts.append(str(caption))
            owners.append(index)
    return texts, owners


def _ranks_i2t(scores: np.ndarray, owners: Sequence[int]) -> list[int]:
    """Per image, the 1-based rank of its best-ranked own caption."""
    owners_arr = np.asarray(owners)
    out = []
    for image, row in enumerate(scores):
        order = np.argsort(-row, kind="stable")
        positions = np.nonzero(owners_arr[order] == image)[0]
        out.append(int(positions[0]) + 1)
    return out


def _ranks_t2i(scores: np.ndarray, owners: Sequence[int]) -> list[int]:
    """Per caption, the 1-based rank of its own image."""
    out = []
    for text, owner in enumerate(owners):
        column = scores[:, text]
        order = np.argsort(-column, kind="stable")
        out.append(int(np.nonzero(order == owner)[0][0]) + 1)
    return out


def retrieval_metrics(scores: Any, owners: Sequence[int]) -> dict[str, Any]:
    """Recall@1/5/10 in both directions, median ranks and rsum for an `[image][caption]` score grid whose
    caption `owners` give each caption's image index."""
    grid = np.asarray(scores, dtype=np.float64)
    if grid.ndim != 2 or grid.shape[1] != len(owners):
        raise ValueError(
            f"scores must be an [images x captions] grid matching {len(owners)} owners, got {grid.shape}"
        )
    if not grid.shape[0] or set(owners) != set(range(grid.shape[0])):
        raise ValueError(
            "every image needs at least one caption in the gallery and every owner must be an image"
        )
    if not np.all(np.isfinite(grid)):
        raise ValueError("scores must be finite")
    i2t, t2i = _ranks_i2t(grid, owners), _ranks_t2i(grid, owners)
    out: dict[str, Any] = {"n_images": int(grid.shape[0]), "n_captions": int(grid.shape[1])}
    for k in RECALL_KS:
        out[f"i2t_recall_at_{k}"] = sum(r <= k for r in i2t) / len(i2t)
        out[f"t2i_recall_at_{k}"] = sum(r <= k for r in t2i) / len(t2i)
    out["i2t_median_rank"] = float(np.median(i2t))
    out["t2i_median_rank"] = float(np.median(t2i))
    out["rsum"] = sum(out[f"{d}_recall_at_{k}"] for d in ("i2t", "t2i") for k in RECALL_KS)
    return out


def itm_pair_accuracy(positive: Sequence[float], negative: Sequence[float]) -> float:
    """Fraction of pairs where the positive's match probability exceeds the hard negative's."""
    if len(positive) != len(negative) or not positive:
        raise ValueError("positive and negative must be non-empty and parallel")
    return sum(p > n for p, n in zip(positive, negative, strict=True)) / len(positive)


def _expected_recall(n_candidates: int, n_correct: int, k: int) -> float:
    """P(at least one of `n_correct` items lands in a random top-k of `n_candidates`)."""
    k = min(k, n_candidates)
    if n_correct >= n_candidates:
        return 1.0
    return 1.0 - math.comb(n_candidates - n_correct, k) / math.comb(n_candidates, k)


def chance_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """The analytical expectation of a uniformly random ranking over the same gallery."""
    _, owners = gallery(records)
    n_images, n_captions = len(records), len(owners)
    if not n_images:
        raise ValueError("the chance baseline needs records")
    out: dict[str, Any] = {"n_images": n_images, "n_captions": n_captions}
    for k in RECALL_KS:
        out[f"i2t_recall_at_{k}"] = (
            sum(_expected_recall(n_captions, len(r["captions"]), k) for r in records) / n_images
        )
        out[f"t2i_recall_at_{k}"] = min(k, n_images) / n_images
    out["i2t_median_rank"] = None  # no closed form worth stating for several correct captions per image
    out["t2i_median_rank"] = (n_images + 1) / 2
    out["rsum"] = sum(out[f"{d}_recall_at_{k}"] for d in ("i2t", "t2i") for k in RECALL_KS)
    out["baseline"] = "chance (analytical expectation of a random ranking)"
    return out


def colour_signature(image: str | Path | Image.Image, *, grid: int = COLOUR_GRID) -> list[float]:
    """Mean RGB of each cell of a `grid` x `grid` partition of the image, in 0..1 (27 numbers by default)."""
    handle = image if isinstance(image, Image.Image) else Image.open(image)
    with handle:
        small = handle.convert("RGB").resize((grid * 8, grid * 8), Image.BILINEAR)
        pixels = list(small.getdata())
    out: list[float] = []
    for row in range(grid):
        for col in range(grid):
            cell = [pixels[(row * 8 + y) * grid * 8 + col * 8 + x] for y in range(8) for x in range(8)]
            out.extend(sum(p[channel] for p in cell) / (64 * 255.0) for channel in range(3))
    return out


def _distance(a: Sequence[float], b: Sequence[float]) -> float:
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b, strict=True)))


def colour_keyword_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """A non-neural score grid over `records`: text → image through the query's closest training caption
    and that photograph's colour; image → text through the colour-nearest training photograph's captions.
    The two directions use different scores, so the grid is assembled per direction."""
    if not train:
        raise ValueError("the colour-keyword baseline needs training records")
    train_sig = [colour_signature(r["image"]) for r in train]
    train_caps = [(str(c), i) for i, r in enumerate(train) for c in r["captions"]]
    texts, owners = gallery(records)
    test_sig = [colour_signature(r["image"]) for r in records]
    # text -> image: score[image, caption] = -distance(image colour, colour of the caption's nearest
    # training image)
    t2i = np.zeros((len(records), len(texts)))
    for j, text in enumerate(texts):
        best = max(train_caps, key=lambda tc: unigram_f1(text, tc[0]))[1]
        for i, sig in enumerate(test_sig):
            t2i[i, j] = -_distance(sig, train_sig[best])
    # image -> text: score[image, caption] = max F1 between the caption and the captions of the image's
    # colour twin
    i2t = np.zeros_like(t2i)
    for i, sig in enumerate(test_sig):
        twin = min(range(len(train)), key=lambda t: _distance(sig, train_sig[t]))
        twin_caps = [str(c) for c in train[twin]["captions"]]
        for j, text in enumerate(texts):
            i2t[i, j] = max(unigram_f1(text, c) for c in twin_caps)
    m_t2i, m_i2t = retrieval_metrics(t2i, owners), retrieval_metrics(i2t, owners)
    out: dict[str, Any] = {"n_images": len(records), "n_captions": len(texts)}
    for k in RECALL_KS:
        out[f"i2t_recall_at_{k}"] = m_i2t[f"i2t_recall_at_{k}"]
        out[f"t2i_recall_at_{k}"] = m_t2i[f"t2i_recall_at_{k}"]
    out["i2t_median_rank"], out["t2i_median_rank"] = m_i2t["i2t_median_rank"], m_t2i["t2i_median_rank"]
    out["rsum"] = sum(out[f"{d}_recall_at_{k}"] for d in ("i2t", "t2i") for k in RECALL_KS)
    out["baseline"] = f"colour-keyword nearest neighbour ({len(train)} training photographs)"
    return out

**Module 2/3:** `src/blip_itm_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Image-text matching and retrieval with the pinned ``Salesforce/blip-itm-base-coco`` checkpoint (BLIP).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the BLIP architecture comes from the pinned ``transformers`` release and no
model-repository code is executed. Upstream ships no SafeTensors at this revision: the PyTorch weights
are ``pytorch_model.bin`` (a pickle), so the trust boundary is the manifest SHA-256 checked before the
load plus ``weights_only=True`` deserialisation; the ``tf_model.h5`` upstream also hosts is the DIMER
upload artifact and is never loaded here. Two scores per image-caption pair: the ITM head's match
probability and the ITC cosine similarity.
"""

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "Salesforce/blip-itm-base-coco"
MODEL_REVISION = "bed8ad38cb2d04a5a4bdf2d071b3c3c0a4aa724c"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "blip-itm-base-coco"
WEIGHT_FILE = "pytorch_model.bin"  # the only PyTorch weight file upstream: a pickle, digest-pinned
HOSTED_TF_WEIGHT_FILE = "tf_model.h5"  # DIMER upload artifact (accepted format); never loaded by this package
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_SHA256 = "017fb3e7f4e125f13a8a4717f1402dbe0d0bb877474b4a203db13a4447b0227f"  # pytorch_model.bin
PARAMETER_COUNT = 223_744_258
TEXT_LAYERS = 12  # text_config num_hidden_layers of the fused text encoder
DEFAULT_TRAINABLE_TEXT_LAYERS = (
    2  # the last two text-encoder blocks + both projections + itm_head (19,298,818 params)
)
ITC_TEMPERATURE = 0.07  # BLIP's contrastive temperature (fixed; the HF checkpoint carries none)
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.blip-itm-base-coco.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"

# Grid ceilings. Every image-caption pair costs one fused forward pass (ITM) plus one dual-encoder pass
# (ITC), so the grid is bounded; a caption is one sentence for the BERT tokenizer.
MAX_IMAGES = 16
MAX_TEXTS = 16
MAX_TEXT_CHARS = 256
# Input ceilings. The processor resizes every image to 384x384 (preprocessor_config.json, aspect
# ratio not preserved) into 24x24 = 576 ViT-B/16 patches, so image cost is bounded; the side ceiling
# only guards memory during decoding and resizing.
IMAGE_SIZE = 384
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def format_texts(texts: Sequence[str]) -> list[str]:
    """Validate a list of captions: str, non-empty after whitespace collapse, within the ceiling, distinct."""
    if isinstance(texts, str) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a list of captions, not a single string")
    if not 1 <= len(texts) <= MAX_TEXTS:
        raise ValueError(f"caption count {len(texts)} outside 1..MAX_TEXTS {MAX_TEXTS}")
    cleaned: list[str] = []
    for text in texts:
        if not isinstance(text, str):
            raise TypeError(f"caption must be str, got {type(text).__name__}")
        collapsed = " ".join(text.split())
        if not collapsed:
            raise ValueError("captions must not be empty")
        if len(collapsed) > MAX_TEXT_CHARS:
            raise ValueError(
                f"caption {collapsed[:12]!r}... is {len(collapsed)} chars > MAX_TEXT_CHARS {MAX_TEXT_CHARS}"
            )
        cleaned.append(collapsed)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("captions must be distinct after whitespace normalisation")
    return cleaned


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def validate_images(images: Any) -> list[Image.Image]:
    if isinstance(images, Image.Image) or not isinstance(images, Sequence) or not images:
        raise TypeError("images must be a non-empty sequence of PIL.Image.Image")
    if len(images) > MAX_IMAGES:
        raise ValueError(f"image count {len(images)} > MAX_IMAGES {MAX_IMAGES}")
    return [validate_image(image) for image in images]


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "1..MAX_IMAGES PIL.Image.Image (any mode, converted to RGB) and 1..MAX_TEXTS caption strings; "
        "every image-caption pair is scored"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "images": [1, MAX_IMAGES],
    "texts": [1, MAX_TEXTS],
    "text_chars": [1, MAX_TEXT_CHARS],
    "preprocessing": (
        f"image resized to {IMAGE_SIZE}x{IMAGE_SIZE} (aspect ratio not preserved, CLIP mean/std) into 576 "
        "ViT-B/16 patches; caption tokenised by the snapshot's BERT tokenizer; per pair the ITC cosine "
        "similarity of the projected image and text embeddings and the ITM head's match/no-match logits "
        "over the fused representation"
    ),
    "output": (
        "per image-caption pair: itm_probability (softmax over the ITM head's two logits, a relative "
        "match score, not calibrated), the raw ITM logits and the ITC cosine similarity; per image the "
        "captions ranked by itm_probability"
    ),
}


def _check_inputs(images: Any, texts: Any) -> tuple[list[Image.Image], list[str]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``score`` and ``validate_inputs`` both route through this function so their acceptance criteria
    cannot diverge.
    """
    return validate_images(images), format_texts(texts)


def validate_inputs(
    images: Sequence[Image.Image],
    texts: Sequence[str],
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every image and caption is checked exactly as ``score`` would check it; rejection is reported by
    raising, and a caller that wants the finding recorded catches the exception and stores ``str(exc)``
    under ``findings``.
    """
    rgb, captions = _check_inputs(images, texts)
    if names is not None and len(names) != len(rgb):
        raise ValueError(f"names has {len(names)} entries for {len(rgb)} images")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[index] if names else f"image-{index}", "mode": image.mode, "size": list(image.size)}
            for index, image in enumerate(images)
        ],
        "texts": captions,
        "n_pairs": len(rgb) * len(captions),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def recall_at_1(scores: np.ndarray, correct: Sequence[int]) -> float:
    """Fraction of rows whose highest-scoring column is the labelled one (a square or rectangular grid)."""
    grid = np.asarray(scores, dtype=np.float64)
    if grid.ndim != 2 or grid.shape[0] != len(correct):
        raise ValueError(f"scores must be a 2-D grid with one correct column per row, got {grid.shape}")
    hits = 0
    for row, target in zip(grid, correct, strict=True):
        if isinstance(target, bool) or not isinstance(target, int) or not 0 <= target < grid.shape[1]:
            raise ValueError("each correct entry must be a valid zero-based column index")
        hits += int(np.argmax(row)) == target
    return hits / grid.shape[0]


def evaluation_report(
    result: Mapping[str, Any],
    correct_text_per_image: Sequence[int] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``correct_text_per_image`` (the zero-based index of each image's matching caption, in image
    order; a one-to-one grid is assumed for the text-to-image direction) the report carries image-to-text
    and text-to-image ``recall_at_1`` for both the ITM probability and the ITC cosine grid, the chance
    baseline, and the verdict ``sample-sanity``; without it the report is ``not-measurable`` and says what
    labelled data would make the task measurable.
    """
    itm = np.asarray(result["itm_probability"], dtype=np.float64)
    cosine = np.asarray(result["cosine"], dtype=np.float64)
    n_images, n_texts = itm.shape
    base = {
        "task": "image-text matching / retrieval over a caller-supplied grid of images and captions",
        "score_semantics": (
            "itm_probability is the softmax of the ITM head's match/no-match logits per pair (a relative "
            "match score, not calibrated, independent across pairs); cosine is the ITC similarity of the "
            "projected embeddings (comparable within a row or column, not a probability); no abstention"
        ),
        "sample_kind": sample_kind,
        "n_images": int(n_images),
        "n_texts": int(n_texts),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if correct_text_per_image is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no image-caption correspondence was supplied for the scored grid",
            "needs": (
                "captioned images from the deployment domain (COCO/Flickr30k-style, several captions per "
                "image) scored with recall@1/5/10 in both directions over thousands of candidates; no such "
                "labelled set ships with this repository"
            ),
        }
    if len(correct_text_per_image) != n_images:
        raise ValueError(
            f"correct_text_per_image has {len(correct_text_per_image)} entries for {n_images} images"
        )
    correct_image_per_text: list[int] | None = None
    if n_images == n_texts and sorted(correct_text_per_image) == list(range(n_texts)):
        inverse = {text: image for image, text in enumerate(correct_text_per_image)}
        correct_image_per_text = [inverse[text] for text in range(n_texts)]
    metrics = []
    for score_id, grid in (("itm_probability", itm), ("cosine", cosine)):
        metrics.append(
            {
                "id": f"image_to_text_recall_at_1_{score_id}",
                "value": recall_at_1(grid, correct_text_per_image),
                "estimation": f"{n_images} image(s) against {n_texts} caption(s), no dispersion estimate",
            }
        )
        if correct_image_per_text is not None:
            metrics.append(
                {
                    "id": f"text_to_image_recall_at_1_{score_id}",
                    "value": recall_at_1(grid.T, correct_image_per_text),
                    "estimation": f"{n_texts} caption(s) against {n_images} image(s), no dispersion estimate",
                }
            )
    return {
        **base,
        "metrics": metrics,
        "baselines": [
            {"id": "chance_image_to_text", "value": 1.0 / n_texts, "note": "random pick among the captions"},
            {"id": "chance_text_to_image", "value": 1.0 / n_images, "note": "random pick among the images"},
        ],
        "verdict": "sample-sanity",
        "reason": (
            f"a {n_images}x{n_texts} grid of images and captions you drew and wrote yourself; plumbing "
            "evidence, not a retrieval benchmark"
        ),
        "needs": (
            "a captioned image set from the deployment domain with thousands of candidates for any "
            "recall@k claim; COCO and Flickr30k are not bundled"
        ),
    }


@dataclass
class BlipItmPipeline:
    """``_runner(image, text)`` returns ``{"itm_logits": [no_match, match], "cosine": float}`` per pair;
    injectable so the offline tests run without the model."""

    _runner: Callable[[Image.Image, str], dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BlipItmPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import BlipForImageTextRetrieval, BlipProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = BlipProcessor.from_pretrained(location, **common)
        # Trust boundary: the only PyTorch weight file upstream is a pickle (pytorch_model.bin). Its
        # SHA-256 was checked against the manifest above; use_safetensors=False names that fact, and
        # weights_only=True makes transformers deserialise with torch.load(weights_only=True), whose
        # restricted unpickler admits tensors, primitives and containers only.
        model = BlipForImageTextRetrieval.from_pretrained(
            location, dtype=torch.float32, use_safetensors=False, weights_only=True, **common
        )
        model = model.eval().to(resolved_device)
        for param in model.parameters():
            param.requires_grad_(False)

        def runner(image: Image.Image, text: str) -> dict[str, Any]:
            model_device = next(model.parameters()).device
            inputs = processor(images=image, text=text, return_tensors="pt").to(model_device)
            with torch.inference_mode():
                itm_logits = model(**inputs)[0]
                cosine = model(**inputs, use_itm_head=False)[0]
            return {
                "itm_logits": [float(v) for v in itm_logits[0].float().cpu().tolist()],
                "cosine": float(cosine.reshape(-1)[0]),
            }

        return cls(runner, resolved_device, "float32", source, _model=model, _processor=processor)

    def score(self, images: Sequence[Image.Image], texts: Sequence[str]) -> dict[str, Any]:
        """Score every image-caption pair; grids are indexed ``[image][text]``."""
        rgb, captions = _check_inputs(images, texts)
        itm_probability = np.zeros((len(rgb), len(captions)), dtype=np.float64)
        itm_logit_match = np.zeros_like(itm_probability)
        cosine = np.zeros_like(itm_probability)
        for i, image in enumerate(rgb):
            for j, caption in enumerate(captions):
                raw = self._runner(image, caption)
                if not isinstance(raw, dict) or "itm_logits" not in raw or "cosine" not in raw:
                    raise RuntimeError("runner must return a dict with 'itm_logits' and 'cosine'")
                logits = np.asarray(raw["itm_logits"], dtype=np.float64).reshape(-1)
                if logits.shape != (2,) or not np.all(np.isfinite(logits)):
                    raise RuntimeError(f"runner returned malformed ITM logits {raw['itm_logits']!r}")
                shifted = np.exp(logits - logits.max())
                itm_probability[i, j] = float(shifted[1] / shifted.sum())
                itm_logit_match[i, j] = float(logits[1])
                cosine[i, j] = float(raw["cosine"])
        rankings = [
            [
                {
                    "text": captions[j],
                    "itm_probability": float(itm_probability[i, j]),
                    "cosine": float(cosine[i, j]),
                }
                for j in np.argsort(-itm_probability[i])
            ]
            for i in range(len(rgb))
        ]
        return {
            "itm_probability": itm_probability,
            "itm_logit_match": itm_logit_match,
            "cosine": cosine,
            "rankings": rankings,
            "texts": captions,
            "n_images": len(rgb),
            "image_sizes": [list(image.size) for image in rgb],
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation contract ---------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._processor

    def _image_embeds(self, paths: Sequence[str], *, batch_size: int = 8) -> dict[str, Any]:
        """The frozen vision encoder's full output per image path (computed once, kept on the model
        device): the ITM head cross-attends to every patch, the ITC branch uses the class token."""
        import torch

        model, processor = self._require_model()
        device = next(model.parameters()).device
        out: dict[str, Any] = {}
        unique = list(dict.fromkeys(paths))
        for start in range(0, len(unique), batch_size):
            chunk = unique[start : start + batch_size]
            images = []
            for path in chunk:
                with Image.open(path) as image:
                    images.append(image.convert("RGB"))
            pixel_values = processor(images=images, return_tensors="pt")["pixel_values"].to(device)
            with torch.inference_mode():
                embeds = model.vision_model(pixel_values=pixel_values)[0]
            for path, embed in zip(chunk, embeds, strict=True):
                out[path] = embed.detach().clone()
        return out

    def _text_features(self, texts: Sequence[str], *, batch_size: int = 32) -> Any:
        """Normalised ITC text features (no image), `[len(texts), proj]` on the model device."""
        import torch
        from torch.nn.functional import normalize

        model, processor = self._require_model()
        device = next(model.parameters()).device
        chunks = []
        for start in range(0, len(texts), batch_size):
            tokens = processor.tokenizer(
                list(texts[start : start + batch_size]), padding=True, return_tensors="pt"
            ).to(device)
            with torch.inference_mode():
                hidden = model.text_encoder(
                    input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"]
                )[0]
                chunks.append(normalize(model.text_proj(hidden[:, 0, :]), dim=-1))
        return torch.cat(chunks)

    def _image_features(self, embeds: Sequence[Any]) -> Any:
        """Normalised ITC image features from cached vision outputs."""
        import torch
        from torch.nn.functional import normalize

        model, _ = self._require_model()
        stacked = torch.stack([e[0] for e in embeds])
        with torch.inference_mode():
            return normalize(model.vision_proj(stacked), dim=-1)

    def _itm_probabilities(self, pairs: Sequence[tuple[Any, str]], *, batch_size: int = 16) -> list[float]:
        """ITM match probability per (cached image embedding, caption) pair."""
        import torch

        model, processor = self._require_model()
        device = next(model.parameters()).device
        out: list[float] = []
        for start in range(0, len(pairs), batch_size):
            chunk = pairs[start : start + batch_size]
            tokens = processor.tokenizer([t for _e, t in chunk], padding=True, return_tensors="pt").to(device)
            image_embeds = torch.stack([e for e, _t in chunk])
            image_atts = torch.ones(image_embeds.shape[:2], dtype=torch.long, device=device)
            with torch.inference_mode():
                hidden = model.text_encoder(
                    input_ids=tokens["input_ids"],
                    attention_mask=tokens["attention_mask"],
                    encoder_hidden_states=image_embeds,
                    encoder_attention_mask=image_atts,
                )[0]
                logits = model.itm_head(hidden[:, 0, :])
                out.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
        return out

    def evaluate(self, records: Sequence[Mapping[str, Any]], *, rerank_top_k: int = 0) -> dict[str, Any]:
        """Retrieval over a validated dataset: every caption of every record is the gallery, the ITC cosine
        grid gives recall@1/5/10 in both directions and rsum; with `rerank_top_k` > 0 the ITM head re-ranks
        the top candidates of every photograph (`itm_i2t_recall_at_1`) and of every photograph's query caption
        — its first caption, one per photograph (`itm_t2i_recall_at_1`) — and the ITM pair accuracy against
        the hardest ITC negative is reported."""

        pass  # standalone rewrite (build_notebook.py): `from .metrics import gallery, itm_pair_accuracy, retrieval_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(rerank_top_k, bool) or not isinstance(rerank_top_k, int) or not 0 <= rerank_top_k <= 50:
            raise ValueError("rerank_top_k must be an int in 0..50")
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        self._require_model()  # before any torch import: an injected runner cannot build the grid
        started = time.perf_counter()
        texts, owners = gallery(checked)
        embeds = self._image_embeds([r["image"] for r in checked])
        ordered = [embeds[r["image"]] for r in checked]
        image_feat = self._image_features(ordered)
        text_feat = self._text_features(texts)
        cosine = (image_feat @ text_feat.T).float().cpu().numpy()
        metrics = retrieval_metrics(cosine, owners)
        metrics["score"] = "itc_cosine"
        categories = sorted({str(r["category"]) for r in checked})
        if len(categories) > 1:
            # each category scored as its own sub-gallery (its photographs against their captions)
            by_category = {}
            for category in categories:
                images = [i for i, r in enumerate(checked) if str(r["category"]) == category]
                members = set(images)
                texts_idx = [j for j, o in enumerate(owners) if o in members]
                remap = {image: k for k, image in enumerate(images)}
                sub = retrieval_metrics(
                    cosine[np.ix_(images, texts_idx)], [remap[owners[j]] for j in texts_idx]
                )
                by_category[category] = {
                    "n": len(images),
                    "i2t_recall_at_1": sub["i2t_recall_at_1"],
                    "t2i_recall_at_1": sub["t2i_recall_at_1"],
                    "rsum": sub["rsum"],
                }
            metrics["by_category"] = by_category
        if rerank_top_k:
            k = min(rerank_top_k, len(texts))
            owners_arr = np.asarray(owners)
            # image -> text: rerank each image's top-k captions
            pairs, index = [], []
            for i in range(len(checked)):
                top = np.argsort(-cosine[i], kind="stable")[:k]
                for j in top:
                    pairs.append((ordered[i], texts[int(j)]))
                    index.append((i, int(j)))
            probs = self._itm_probabilities(pairs)
            hits_i2t = 0
            best_by_image: dict[int, tuple[float, int]] = {}
            for (i, j), p in zip(index, probs, strict=True):
                if i not in best_by_image or p > best_by_image[i][0]:
                    best_by_image[i] = (p, j)
            hits_i2t = sum(owners_arr[j] == i for i, (_p, j) in best_by_image.items())
            # text -> image: rerank each query caption's top-k images (the first caption of every record)
            k_img = min(rerank_top_k, len(checked))
            queries = [next(j for j, o in enumerate(owners) if o == i) for i in range(len(checked))]
            pairs, index = [], []
            for j in queries:
                top = np.argsort(-cosine[:, j], kind="stable")[:k_img]
                for i in top:
                    pairs.append((ordered[int(i)], texts[j]))
                    index.append((int(i), j))
            probs = self._itm_probabilities(pairs)
            best_by_text: dict[int, tuple[float, int]] = {}
            for (i, j), p in zip(index, probs, strict=True):
                if j not in best_by_text or p > best_by_text[j][0]:
                    best_by_text[j] = (p, i)
            hits_t2i = sum(owners[j] == i for j, (_p, i) in best_by_text.items())
            n_queries = len(queries)
            # pair accuracy: query caption vs the hardest wrong caption by ITC
            pairs = []
            for i, record in enumerate(checked):
                row = cosine[i].copy()
                row[owners_arr == i] = -np.inf
                hardest = int(np.argmax(row))
                pairs.append((ordered[i], str(record["captions"][0])))
                pairs.append((ordered[i], texts[hardest]))
            probs = self._itm_probabilities(pairs)
            metrics.update(
                {
                    "itm_rerank_top_k": rerank_top_k,
                    "itm_i2t_recall_at_1": float(hits_i2t) / len(checked),
                    "itm_t2i_recall_at_1": float(hits_t2i) / n_queries,
                    "itm_t2i_queries": n_queries,
                    "itm_pair_accuracy": itm_pair_accuracy(probs[0::2], probs[1::2]),
                }
            )
        metrics.update(
            {
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_text_layers: int) -> list[str]:
        """The last `trainable_text_layers` blocks of the fused text encoder (self-attention, cross-attention
        and feed-forward), both ITC projections and the ITM head. The vision encoder and the text embeddings
        stay frozen."""
        if (
            isinstance(trainable_text_layers, bool)
            or not isinstance(trainable_text_layers, int)
            or not 1 <= trainable_text_layers <= TEXT_LAYERS
        ):
            raise ValueError(f"trainable_text_layers must be an int in 1..{TEXT_LAYERS}")
        model, _ = self._require_model()
        first = TEXT_LAYERS - trainable_text_layers
        prefixes = tuple(f"text_encoder.encoder.layer.{k}." for k in range(first, TEXT_LAYERS)) + (
            "vision_proj.",
            "text_proj.",
            "itm_head.",
        )
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 4,
        lr: float = 2e-5,
        batch_size: int = 16,
        trainable_text_layers: int = DEFAULT_TRAINABLE_TEXT_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on a validated image-caption dataset.

        Only the last `trainable_text_layers` blocks of the fused text encoder, the two ITC projections and
        the ITM head train (2 blocks by default: 19,298,818 of 223,744,258 parameters; the vision encoder and
        the text embeddings stay frozen). The frozen vision encoder's output is computed once per training
        photograph and reused across epochs. Every (photograph, caption) pair is one sample; a batch of
        pairs trains BLIP's two objectives: the image-text contrastive loss (symmetric cross-entropy over the
        in-batch cosine similarities at temperature 0.07, pairs of the same photograph counted as positives)
        and the image-text matching loss (binary cross-entropy of the ITM head on the batch's positives plus
        one hard negative caption per photograph and one hard negative photograph per caption, sampled in
        proportion to their ITC similarity, never from the same photograph). AdamW at a fixed learning rate
        with gradient clipping at 1.0, no scheduler. Epoch 0 records the frozen model's validation retrieval
        metrics; the epoch with the highest validation rsum (ITC cosine) is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 2 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 2..64")
        names = self._trainable_names(trainable_text_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        import torch
        from torch.nn.functional import cross_entropy, log_softmax, normalize

        torch.manual_seed(seed)
        model, processor = self._require_model()
        tokenizer = processor.tokenizer
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = next(model.parameters()).device
        embeds = self._image_embeds([r["image"] for r in train_checked])
        pairs = [(i, str(c)) for i, r in enumerate(train_checked) for c in r["captions"]]

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            keep = {"rsum", "n_images", "n_captions"} | {
                f"{d}_recall_at_{k}" for d in ("i2t", "t2i") for k in (1, 5, 10)
            }
            return {k: v for k, v in self.evaluate(val_checked).items() if k in keep}

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_score = entry["val"]["rsum"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(pairs), generator=generator).tolist()
                losses = []
                for start in range(0, len(order) - 1, batch_size):
                    chosen = [pairs[j] for j in order[start : start + batch_size]]
                    if len(chosen) < 2:
                        continue
                    image_index = torch.tensor([i for i, _c in chosen], device=device)
                    image_embeds = torch.stack([embeds[train_checked[i]["image"]] for i, _c in chosen])
                    tokens = tokenizer([c for _i, c in chosen], padding=True, return_tensors="pt").to(device)
                    # ITC: dual encoders, symmetric contrastive loss with same-photograph pairs as positives
                    text_hidden = model.text_encoder(
                        input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"]
                    )[0]
                    text_feat = normalize(model.text_proj(text_hidden[:, 0, :]), dim=-1)
                    image_feat = normalize(model.vision_proj(image_embeds[:, 0, :]), dim=-1)
                    sim = image_feat @ text_feat.T / ITC_TEMPERATURE
                    same = (image_index[:, None] == image_index[None, :]).float()
                    targets = same / same.sum(dim=1, keepdim=True)
                    loss_itc = (
                        -(
                            (targets * log_softmax(sim, dim=1)).sum(dim=1).mean()
                            + (targets.T * log_softmax(sim.T, dim=1)).sum(dim=1).mean()
                        )
                        / 2
                    )
                    # ITM: positives + one hard negative caption per photograph + one hard negative
                    # photograph per caption, sampled in proportion to the (detached) ITC similarity
                    with torch.no_grad():
                        weights = torch.softmax(sim.detach().float(), dim=1) * (1 - same) + 1e-6
                        neg_text = torch.multinomial(weights, 1).squeeze(1)
                        weights_t = torch.softmax(sim.detach().float().T, dim=1) * (1 - same) + 1e-6
                        neg_image = torch.multinomial(weights_t, 1).squeeze(1)
                    itm_images = torch.cat([image_embeds, image_embeds, image_embeds[neg_image]])
                    itm_ids = torch.cat(
                        [tokens["input_ids"], tokens["input_ids"][neg_text], tokens["input_ids"]]
                    )
                    itm_mask = torch.cat(
                        [
                            tokens["attention_mask"],
                            tokens["attention_mask"][neg_text],
                            tokens["attention_mask"],
                        ]
                    )
                    image_atts = torch.ones(itm_images.shape[:2], dtype=torch.long, device=device)
                    fused = model.text_encoder(
                        input_ids=itm_ids,
                        attention_mask=itm_mask,
                        encoder_hidden_states=itm_images,
                        encoder_attention_mask=image_atts,
                    )[0]
                    logits = model.itm_head(fused[:, 0, :])
                    labels = torch.cat(
                        [
                            torch.ones(len(chosen), dtype=torch.long, device=device),
                            torch.zeros(2 * len(chosen), dtype=torch.long, device=device),
                        ]
                    )
                    loss_itm = cross_entropy(logits, labels)
                    loss = loss_itc + loss_itm
                    optimiser.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["rsum"] if entry["val"] else math.inf
                if current > best_score or not entry["val"]:
                    best_score = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_text_layers": trainable_text_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation rsum (ITC cosine)"
            if val_checked
            else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "itc_temperature": ITC_TEMPERATURE,
            "n_train": len(train_checked),
            "n_pairs": len(pairs),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted text-encoder-block, projection and ITM-head tensors as safetensors plus a base
        manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported format
        and version, the pinned base (id, revision, weight file, digest), exactly one file entry named
        `adapter.safetensors` that resolves inside the artifact directory, and a recorded
        `trainable_text_layers` in range. Nothing is deserialised here. The digest check that follows
        detects corruption or drift of the weights relative to the adjacent manifest; it is not authenticity
        against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHT_FILE) != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_text_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int) or not 1 <= layers <= TEXT_LAYERS:
            raise ValueError("artifact manifest does not record an in-range integer trainable_text_layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_text_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        allowed = ("text_encoder.encoder.layer.", "vision_proj.", "text_proj.", "itm_head.")
        for key, value in tensors.items():
            if key not in state or not key.startswith(allowed):
                raise ValueError(f"artifact tensor {key} is not an adaptable tensor of the base")
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BlipItmPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/blip_itm_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Image-text dataset contract for retrieval fine-tuning: the pinned VizWiz-Captions sample, validation,
seeded image-disjoint splitting, BYOD loaders and JSONL export.

The default dataset is **real** and out of the base model's distribution: VizWiz-Captions (Gurari et al.,
ECCV 2020; CC BY 4.0) — photographs taken by blind people, each with five crowd-written captions, from a
population and a caption style (what is held, what the label says, how the picture is framed) that COCO
captions do not cover. `Salesforce/blip-itm-base-coco` was fine-tuned on COCO retrieval and never on VizWiz.
The sample comes from one pinned parquet shard of the Hub mirror `mm-eval/VizWiz-Captions` (a re-conversion
of the official `val.json` with the rejected and pre-canned captions dropped, so an image can carry fewer
than five references): the four text columns of all 1,550 rows are read **column-only** over HTTPS range
requests through `pyarrow` (about 0.5 MB), and the image column of **row group 0 only** (336 photographs,
about 84 MB) is read the same way — the shard's declared size and SHA-256 checked against the pins before
any byte is read, the decoded text columns' SHA-256 and every photograph's SHA-256 and size checked after.
Captions longer than the scoring ceiling `MAX_TEXT_CHARS` are dropped (one of 1,391) and images without a
surviving caption are excluded from the sample.

A record is ``{id, image_id, image, captions, category}`` — the path of the digest-verified photograph, its
one or more matching captions (every one a positive pair; the first is the retrieval query) and a category
(`text` when the VizWiz annotators flagged text in the photograph, `no-text` otherwise; BYOD records may
carry any label, `other` by default). Every record's image is its own (one row per photograph), so a split
by image is a split by record; BYOD records may share an `image_id` and are then kept together.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import re
import urllib.request
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TEXT_CHARS, MODEL_ID, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "VizWiz-Captions"
CORPUS_REPO = "mm-eval/VizWiz-Captions"
CORPUS_REVISION = "c4a6d897836e7885d0095134f92d392e4e770539"
CORPUS_RELEASE = (
    "VizWiz-Captions v1 (2020) as re-converted on the Hugging Face Hub, dataset revision c4a6d897"
)
CORPUS_LICENSE = "CC BY 4.0 (Gurari et al. 2020; vizwiz.org/tasks-and-datasets/image-captioning)"
CORPUS_TEXT_COLUMNS = ("id", "answer", "question_type", "text_detected")
CORPUS_IMAGE_COLUMN = "media"
CORPUS_FILE: dict[str, Any] = {
    "path": "data/val-00004-of-00005.parquet",
    "bytes": 392_245_504,
    "sha256": "4492465a41d32b3c12b8b7b6a0cf7e0a0e202a5b825b006ca0c85dcdf24efd3e",
    "rows": 1_550,
    "text_sha256": "9799ebb13cf6a7e7c76afdd180892e21499ecefff697212ce4e505c7fc207d6e",
    "row_groups": [0, 1],
    "row_group_rows": [336, 336],
}
DEFAULT_CACHE_DIR = Path("weights") / "vizwiz-captions"
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 208, "validation": 40, "test": 70}  # the 318 row-group-0 photographs with a caption
GALLERY_ROW_GROUP = 1  # every captioned photograph of this row group joins the test gallery (321 of 336)
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MIN_CAPTIONS = 1
MAX_CAPTION_CHARS = MAX_TEXT_CHARS  # a caption must be scorable by `score`
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
# image id -> (sha256, bytes, row group) of the JPEG carried by the pinned shard: row group 0 (336
# photographs: training / validation / core test) and row group 1 (336 photographs: test gallery only).
IMAGE_PINS: dict[str, tuple[str, int, int]] = {
    "29631": ("635042d060e1f42bdebf7d16af7b7b1f78763482b21959dd6ab3e642fb131522", 219537, 0),
    "29632": ("595732108a90de2ebb653d8bac8ab1d88d837624103c0a3344161f57ff84bd7b", 208376, 0),
    "29633": ("93ff2bd506c778c53715a965a83d8d55a6d3a46b4b06dd91108689284ba3aa5f", 14682, 0),
    "29634": ("b7bfa7344d17b18dcc03f3fdcecb874963a3584d14b92210717f351155e42d0a", 232493, 0),
    "29635": ("f807878774e35d16396c8d6600c67b8bd6cb93a89328da8d29bfd8c34386ec85", 268910, 0),
    "29636": ("fa7f75a23aaedfeb26032a7bce86bdfaace2d204695efafc17b87a0907bae611", 82416, 0),
    "29637": ("407efaaabffc133f142c0c690318a16812c39fc30902ac95d86b3b8640b511c9", 506359, 0),
    "29638": ("486f02412386b81757936e3e9782efe234fe69760d24fc70734a0b60d7b284b0", 443910, 0),
    "29639": ("37cfae24bbfb0faf1d6fadda974b2824bf8976ffe7fac02f740eee75027d95b3", 349992, 0),
    "29640": ("da467dffb38a6d407cee8fe1b1b690a3d8af70f378941e0dd992284f9d2d8f5e", 269887, 0),
    "29641": ("4f2e2bea724a6b3b3ead65718da3b3d17bcd34f56913c0652ba91d89acd6967e", 257024, 0),
    "29642": ("afbdef671c1faa2f191466f7320b07a18c583afe2c3e856eccefec46532f29db", 225839, 0),
    "29643": ("cdb9b4f804e4ee61754d82560940c72fe21a3747fac6c75648faa613231da76a", 261053, 0),
    "29644": ("632a0a1662b58ad6b13cefef34a51898b93fcdf88abebc9217e312691128f569", 193354, 0),
    "29645": ("889242e35ee151e13fe6243567126d0476c9221a3985ecf8dce4f6b415f2b59e", 258707, 0),
    "29646": ("0454ddcec779bfbfafd093518f67165b2a7a092e81917e78c213aa18f6dbb8cb", 346643, 0),
    "29647": ("d0ef537515306a3df5c77d85dd0d2666c4aa0da2f72502cc82500cf6524d1e19", 271956, 0),
    "29648": ("ceb4c0b7b00bbb1dfbdacec67ea4d8fc8d59754a9856c0eb87a52a554a570664", 321526, 0),
    "29649": ("dccb89f70f8be06bdcff3e4947422fb5a69cf5069b00e707c22e92090fe91397", 272027, 0),
    "29650": ("f5e41db77978764a79010a335c6e21d5a053892517dd34a4dfd315ae17eeeb1a", 434003, 0),
    "29651": ("72be2c699582d963754a4986c56c22701511cef3c802c48e0d7aa558f9107a09", 292153, 0),
    "29652": ("7db7f94532355552e80fbeb015d9e4db978398cc57512a9c96bd0db44a50c0c4", 338389, 0),
    "29653": ("0a84818b7f47bf36c5e6e4001481dfc8faec6bf232792ce521b75db2cf50ec86", 188622, 0),
    "29654": ("15eca622fe300527a6c92df119b8a2f3947df676d627047ea50e96de39303d19", 49771, 0),
    "29655": ("b2e54c27133737e21ca33e858ef52dd66b7a68bd9144f947094702295511a891", 39778, 0),
    "29656": ("719f5852b6092754ac26f3feb6d6d974da19a848146ea3bced950e71dd55cc0b", 144496, 0),
    "29657": ("568dd4519806b64e63d40f063205e485c1fc9f42a6719fff7acbe59d94c01148", 199012, 0),
    "29658": ("dac90383b9f44fbff68426da47687c124f2511152ce4d3e94c307855a43e3871", 337908, 0),
    "29659": ("61be884a769cb69f89e847cb643c4a78c01e747934d56186a53a8b3e6daf244f", 30681, 0),
    "29660": ("08fac9a7c1c30e0a1aa92f7295f3366454d4629a78bd3678b86090ffff8fc373", 205850, 0),
    "29661": ("72e87e97b362c7fe3412676ca230cb5765cd6d9091d530e4f1f894458344c97d", 330913, 0),
    "29662": ("61bb302f64ec861c9628b1efc04c065a1fffe4c9fa6688b16d06ce946eec0c1f", 182155, 0),
    "29663": ("a417180d3f937c01b9269cab5e46e096a05714c0c2fdf630bf7430cb351e44e9", 258175, 0),
    "29664": ("3913b82cecc6a49c057c9cc4dde219920f62aa9c43f2b1843c58cebdefa5ee20", 217507, 0),
    "29665": ("e4b02231db046fc3b6b25cfd434ba4ed25ba74f98941cedc7127fa3d5579c0e5", 673261, 0),
    "29666": ("1d6e584c39a7b469a97f7125ab91dbfed50ce151c76c0037365754b3da1935ec", 415821, 0),
    "29667": ("994e2c3cf86c546c10480b01064a9af915add972553ad8b4645a3e5a183969fe", 459367, 0),
    "29668": ("f6c0fc8835171e91cf8368be8c33e8af565c037e89fdd61293e92dc61f7a57ca", 283625, 0),
    "29669": ("3a465d5426ce6c766a9aaee0aa135ff5b438197587a21111bb06f32d827a4229", 155717, 0),
    "29670": ("e60a335c69895e52db1f1265533f773977117819d10cffc7c7a6fe5b77fd911b", 218339, 0),
    "29671": ("45a785cec7d063f545f72b8e5f1c3ce1820c43f62bd725fdaff2070db47468ff", 301005, 0),
    "29672": ("20df609ab7455f25c26a8ad7e79224e921dd3656de31c8e9757b0bcda5a3e6e8", 271522, 0),
    "29673": ("9009def8616940b157d2133306edd1a2f7f39b129a158a9a19ebd88069f246d8", 36083, 0),
    "29674": ("5160d7be46d40cd6cb3b122d94b48e8791ed0a939408884b130b11ce3eff64aa", 311110, 0),
    "29675": ("ea0dcee84b2b3ccaf9371e8ffecc779f6a7acf8a1303b3c3438c3b507a3fde8a", 331823, 0),
    "29676": ("799e06916b505131dbc20db33c6aef34f9c325a347819de063c3faa599c1e607", 209338, 0),
    "29677": ("67fe5425d3f0e7aa2a260d854cf25ea1c1c41f9cb30dc3691b16700966011c85", 176798, 0),
    "29678": ("80727a1e760ecee07214bfec0448fda106fc07f3c849a4248b347514b0fcfa91", 591377, 0),
    "29679": ("80e29692260e0ade581839baec5d5a4550e5822cab0aa15115cd3c96a1569f3e", 33215, 0),
    "29680": ("7c836719c64e5d97ba83b99a6fba966062a180a1e3aaffa30dde4d2117daef89", 373575, 0),
    "29681": ("2416fea174cfd4522953171bbcaa38900aa38d3e9671c6b4ff9dbe314caa5f13", 293047, 0),
    "29682": ("05c5971c974f1c9843ce2bb0d08c1500e380596107ac76a24b057b0efa658573", 466144, 0),
    "29683": ("f3f6e73a1d1dd7ba7449ba0634b09decfd2350dfbe8f3fdcf3fbbdbe2719b5b7", 185222, 0),
    "29684": ("92a3f9f7bcd352a6f1c03d305f98aec1f1d3cc315f3d6c2bab46f13bca7817b6", 325758, 0),
    "29685": ("2db1c9bc24a7a12af81c2d8a5539e422d210e03eda0ecc0af6944e98bdea1305", 304402, 0),
    "29686": ("fd3ebb92d81317be6db45ad6326d3227f095edd2aa93c2f3e571806b5a4301b9", 379396, 0),
    "29687": ("8be2b3264792a2ac22abc9a613d5759e9f31e79b3659f78039a70c08d449860a", 233109, 0),
    "29688": ("d0862ed97636b9cb77577c5ba6a82cb75176548231568a1a50c00bb6c47700d1", 301273, 0),
    "29689": ("31b332bfc7af3903b1f6091cc92627ca299dbe9c2e2330f730228e505e573296", 129869, 0),
    "29690": ("d39dbd910bd9fbe3b36faa99b6580cf371b1aff87230066f91f46d1a55dd124e", 224998, 0),
    "29691": ("3012921370798079afe212db4bc22cfbf284bd7584e74cf33297956bd9189f8e", 385230, 0),
    "29692": ("903718f7a211747b1677bc680f72facb3a4d582abd03131dd98c88ece06d0c54", 280343, 0),
    "29693": ("d44c2877651a8dba7cf2943265f0f6353ff38697cf921fc14c33a5a1afcbcd73", 196010, 0),
    "29694": ("0a20818fe43942fd653591cbed494c38a2550e27e3892753c278142b462f900d", 165487, 0),
    "29695": ("98650214fe93da1f94e6ad9d0fc19bb79e2485223e706e1ba292a50af85c988f", 63581, 0),
    "29696": ("ddfd96ca16ada6259061247926d6bda0d302b3ec7adbda44e66f0a31c27baf11", 490864, 0),
    "29697": ("7e68d03f88d8e6a65ed883ccf45f5b2e1c7b1668e5b168355650311601fb3979", 276114, 0),
    "29698": ("dc8bf976fc17cebf01fbf6779c8400ccf27a12ac7124cf7490a560051c18b50b", 520196, 0),
    "29699": ("54e6417e77612a9b8444e4febb025e12e1488b05aa72fbcf63e8bd69a8bf6652", 52962, 0),
    "29700": ("b03833951a0c568edc3a1568ad14769b58615b57a32d50b297fe12df05989128", 178206, 0),
    "29701": ("d1d5c7795f4c794645bc728e02efe447bc8a4c896aa397fb843925d510262e70", 204765, 0),
    "29702": ("d175d905034ca667379229035051f90abcd49d42d9aedd17cfc19bf6dc4ed7db", 329707, 0),
    "29703": ("429e3c4a9c822847f9670251c8027485505686724caa0cf467fecb4b46afd7e7", 98889, 0),
    "29704": ("1302773b4d1d13c8242e9b6fe283f85bdc6331b01de77f795eded9d2a399d334", 208973, 0),
    "29705": ("beb66040a0e41e4f3ae26db0cb2d78321bfcb71b6f2fe3198671ceda10e3ffa6", 305340, 0),
    "29706": ("ac55c0fd7e334f1a5cdf41006c33cd1c699ac71a9a95556e27f6ea564ae8a587", 260218, 0),
    "29707": ("f86d9785862905e412f37cc2dbd49836c5aa6ead03c586a4f9a76555f1894305", 397788, 0),
    "29708": ("f69d115f7a11814d05ab5e8e2b90070938a71edd890dbc4ae6eb4c62a96d6b8c", 339030, 0),
    "29709": ("dd65b40eee8a23e0bdf47b67d5a3be54b31913ba9918fabe4a9d4aaf068f63aa", 218005, 0),
    "29710": ("741594d8849df32c38f1898498573c18da785e19dea9481723d770b7d47bfad0", 480990, 0),
    "29711": ("b4cb496aa54857053256ea084a4390bc9a157c923ac601c77057ef257439b7d7", 429787, 0),
    "29712": ("8473b6193e761171dd28d3c2cb4894dfeee51b014330c2289cfd40b8d8a53c92", 315871, 0),
    "29713": ("569fa73a25181a278b420556c8bfcce2e2abeb8e36e844c53c977ac8c3aa3e23", 254682, 0),
    "29714": ("c55af0d333f124114fe440ca756b5d39ead1c14b07ebacedbd52300622e46454", 322112, 0),
    "29715": ("d3007697c34a7ef94452e5ef3fd260c549fbfe37473d015c25f880e83f8bd284", 208474, 0),
    "29716": ("a53d0db806788cc105ebd5ab4749eb7a31617a95b0394cc30a808bc5d9f79667", 375964, 0),
    "29717": ("37d835e42fc5693772402122065dbbf8bb7da4642e63c26a97609146dfe64cb2", 30875, 0),
    "29718": ("1682aeae89a9d92c6484dd7ba787922846f463103c4b58fcf3251e91c5190a16", 189651, 0),
    "29719": ("ad695d09ad35c7c2d528d26b30f602e7883e5018007a4d6109c18d3020d3c382", 279840, 0),
    "29720": ("4af2ab447a764814d215712afcbfb12b42de8e9ba29a1aa9b02ef1550240d2d3", 188421, 0),
    "29721": ("574d2175f929da7be1a8d1c2849939be185945aa0565670d8036854e553dd718", 504332, 0),
    "29722": ("88136aa35f1b1bf1ba2caf69fd12c466364712822bace72e5d188f7f0b41ce0b", 234812, 0),
    "29723": ("91f501ed03e17b9860515fb8602956a17d569f927da5b4f61228116b5e282578", 22300, 0),
    "29724": ("81db3e3d3276b80cc910cd207bbe43fe347cd300f52e32f6659cd965590716a7", 173636, 0),
    "29725": ("8901239345b4524db707c92fecb542042d2c44c293ba35efb6b86c60d1feceeb", 267482, 0),
    "29726": ("917694384bbd0debc698d56d6cdd38d18e9a6b4b8d1adef6faf2f4dd3858f83a", 227891, 0),
    "29727": ("864c8141cca16977f599cdcef0d4e1313b96d51044c116f765c7bf226b08468a", 289964, 0),
    "29728": ("2ea6ece248a3fbb0b7a8c0a2143daf36a429c4934b8ee72be014c95a8cecd868", 407475, 0),
    "29729": ("480630ab0b3bc6ebade59d97f2c0f37a99df9b76a88dcf699c22cbfa6b5098eb", 525377, 0),
    "29730": ("ca6c42ddd9afe5d55fdd8f47ea6feb26ba4a5a6600afdf6130b212b352aaf9b6", 96112, 0),
    "29731": ("1cff0ee25e06c7c68ca657cca15d383cc831bcb4734577611afc676eb264d393", 325968, 0),
    "29732": ("f44b95a2c2e643066d75410a7d860d2c12fcd7a890e06d4f9a226044554ceee7", 416985, 0),
    "29733": ("7d72c78e4ca19ac4b71455aa5c7445f1b4f46ceced6580ee57f63345514d7ea4", 456935, 0),
    "29734": ("abee412faa3a0c7c9ec64a65ee533cc717ff5d5aba34532fa1a71c091693dd15", 364966, 0),
    "29735": ("041c0e34e313709ed82a7183eb122e04f588379d70e1688ce4f0fae78748bdca", 5674, 0),
    "29736": ("3837381d2a37dca6c1aff7916a20108ab791a3c609c25adcaaac0ddc4eaed079", 254531, 0),
    "29737": ("a4f9e99838d4cae4fedd6c7327f1f056d0c766ad3b71b3aafd13f1aa2ebd633e", 279112, 0),
    "29738": ("e437e827d89a0e7f948b1130c59e14498b48d794542e6a290f156cdc75e6c8a2", 438833, 0),
    "29739": ("eca903ff72aaa2e6d28f8ba09ebdc11a8870c04ba69d358e4bbdd9c9295cf2e5", 462594, 0),
    "29740": ("aa23104e6d3451495da8f4c4b4697e14195ce91f534b05a07e16df13abca50a8", 209903, 0),
    "29741": ("66b435428d6f56531272b5126d8253d8bf3a5dcebbbe2fede4192a492d531e1a", 396402, 0),
    "29742": ("e06a6f405a309164c69e5b1efe16308cc6a73dd730156f25bf6dcccdde0d40ac", 291934, 0),
    "29743": ("5f500f75a9b89f5046bfe71fac33d3166198f0f54e4fcc8783fbf210c06786cf", 199415, 0),
    "29744": ("370e3f0dfb6fcb8ca4eb460a3f3bbf1a88e6b49ed6e12080043504c1bb583e75", 94905, 0),
    "29745": ("90bc34c278369315792bbdd555b6a28b59c2c05577268cfa8c06fa62634dd61c", 278846, 0),
    "29746": ("ddafbfff05e0bbf6fd951fa47ad23e82221e4bf967f8257f33c0608dd276eecf", 476441, 0),
    "29747": ("fee198facb8b7c32f51d0a71dfedce1b9e5774cfc53292805d46ad07fbcc4c09", 240146, 0),
    "29748": ("d45aeb61c6b1325d67d9e4bae1306b41757e2b6603df737fa4e706aac4c72717", 437285, 0),
    "29749": ("d8d1bea0ad92686a2a81cb9e64b176fcbb42359764037a08e490639ebc0e9101", 378157, 0),
    "29750": ("7c0104f71e089e9f6c3deec5931cc9914ee5cab5e20dadc521d6ab6a9f01d8dc", 153136, 0),
    "29751": ("8110503bbb60bcd584dbb654257501dd629d2cb67ac0c5efba506ef359fd263e", 198878, 0),
    "29752": ("06660e9b67661ddccd6e5f6e29c33b3f7ed277438beb2fe535dac8a4dbc95d75", 178611, 0),
    "29753": ("70d821652187f89d4722c35f2a6fe242ce10fd5ba19d0a20659eb712cc36f7e0", 223117, 0),
    "29754": ("ce8741810135d13631f820b05c6236b5cc59a8c4b32ceb15d954b6c903ed10db", 388709, 0),
    "29755": ("4e48a4ccf5778b08f72e880d1ffcc75c55754e959793b449749f492a4b861fdd", 25809, 0),
    "29756": ("3e8c187144179fe615c4d65f9552796df6db854bbaf452ee04bf2b79a6b46f64", 249967, 0),
    "29757": ("d0c5c9c2841255304d132cfcd21dff3f46a4a1fd949cb46d13290b078c3fc6ad", 404469, 0),
    "29758": ("c8da6129ba6741fe9ab3aec3d1f2a98295a1a3ac0e39e681e0f4c959c50ba57c", 557828, 0),
    "29759": ("ff17439b1a4254ec0ac56e56eeb3a4a79efdab180d3aacb719cada411a86db88", 189077, 0),
    "29760": ("a35bfb5aab5ccda3e08664a1144afc6fe4aacd8d50dadfaad1299176cdba2b81", 41846, 0),
    "29761": ("b605f111974843799b6893c7a676edacedb9c4605a22e9b4bb2481df1685385d", 145351, 0),
    "29762": ("2551862d131925ecc320ca175eb500a6b5f14f034689494521b8f4ad0788d100", 490759, 0),
    "29763": ("a5a65eaa03b4ac5be3d19393de923a3a75a47c0b91e24286ae672d2c8d72410e", 128025, 0),
    "29764": ("4899ac930c06390744285b8eb9d4561e9fdfed9600ff2282d8e88662dd24de57", 299517, 0),
    "29765": ("de3aecd40c5e88d2631d4b6579a6bb05cc3d17a0ca78e4c14b731ce72da9aa7a", 49944, 0),
    "29766": ("8e8192b45a95a651037fe46ba0a376aefc8c416fc613273b451441ddd77f16d8", 10213, 0),
    "29767": ("e1549a8a8a4df8a7adde1d7eda4267141cfea21d500c59585e83633e3c72bfdc", 164821, 0),
    "29768": ("d0df77108a4e1b4c6795ecc2a20b88b278fd2eb1bb1cb2832634ffe212c1601d", 44980, 0),
    "29769": ("1272ee672e32c91ad19f9ccc88bcb0cdd669b4f0a0aa59208c47050f27815b46", 131144, 0),
    "29770": ("1b4b19ebee58cfe4c47c5151d605b478cc25e67cd551d2eb5aec7e9a90850745", 393641, 0),
    "29771": ("6caa00c059b3c0328a4548a205b3df0f48c73ea7b7b3a28481a6be5a133ffe29", 313637, 0),
    "29772": ("58dbfcfd30b1430aa729c308e894f854d3b1760517e57e84fe066310db571806", 45750, 0),
    "29773": ("1a88906df09163fda64f2ab6657d17f40a51bd607c45a54d5e7b1df0fc28f269", 377194, 0),
    "29774": ("aa3f4192358443d4c1a1dafa9df61bbed5d0b4a9819f39666981e05959fc47f6", 311439, 0),
    "29775": ("d9e74a559870f3e71306bf4b308e3b191089a53be2537cd1206091c7496485bb", 422151, 0),
    "29776": ("62fba88862012fccdbe5a546af02ede17a8dae4ef5da0530fd8075b11f7a81ca", 170851, 0),
    "29777": ("a25d5bccdef331fb7d7f4069d4aff6bcb3fd2b5978fcbc786cb63dfc69f35a2f", 192365, 0),
    "29778": ("e6c63a4e9ab1d42cc99b828c401b2df40440f4b98017d73be591cacbbe6d15c6", 35073, 0),
    "29779": ("de04222ff62a9c7d608661adb83048bfa480187d9d370be96ce0c868fb986c73", 353379, 0),
    "29780": ("b18fddc7fb6781eee1362d66f7ebb0b46356a7126228c5a423a6dd2a31b0821e", 191319, 0),
    "29781": ("4e5d80215bcee8645c5a2004fb4e9344b98a1a89eef2cea52b5a391c0471ad23", 7013, 0),
    "29782": ("b810bf11ab589b948767fde0ca1fe359a0fd95cdf479ff562de5e1c2ee08ab91", 177385, 0),
    "29783": ("6a76d12335bc9865e3e6a590e0680a348bed174fe77ee88c17f1b87cc23dc74b", 362709, 0),
    "29784": ("a52552b4845de621251adcbef62ed34ff858bc348bb67957d1ee70a4d8af1c63", 643614, 0),
    "29785": ("b5ba4b328f41ce244cae7d121ea793db6628911c5a19c0236b6600da27e808a0", 438950, 0),
    "29786": ("adad10c2b55b075c88840b83a93d7d9d76191560d250ed50f76c5feea72dbebe", 311019, 0),
    "29787": ("5f34951d0ba8669cee13b5041e3829a0d4add8558c35f286ed364b6645e03509", 408758, 0),
    "29788": ("9dc12128ac87915441da9f6c8e77442b0f8cad7c711d1d292cfaea40fced0819", 510613, 0),
    "29789": ("dc8abffe7b476f2c50800a8ce313a1f9b38590fe772094f7c9913fc00582a02a", 114300, 0),
    "29790": ("87a5d02fd6cf49995ebc927b78165f9b094734a75ec67e33e1403c90d0e0a938", 208011, 0),
    "29791": ("3d7157161b22d66f2f13fbf4bee430ccd0f2b96209ec3294bd81718702ea1ea1", 269717, 0),
    "29792": ("a7b2b807b937331c404f47d7cddedc0294598dc31829c00616cf0859faa2fca3", 248637, 0),
    "29793": ("55f462ab11c6bd2984ba207b28bb3991a613d342924020f36898524d22f42eac", 34816, 0),
    "29794": ("13fe979b44d3dabc50acd949923e90fc92c8155fcb5862072a491b4bccb56702", 231414, 0),
    "29795": ("b0b4e84d43bb1126c219207815189746e573ca79fc6dcd974cb919c98dac6114", 28979, 0),
    "29796": ("e1d375fb269a2f7af483e9642006687bbf48b13de6449eba269ed1ea301ead0b", 309491, 0),
    "29797": ("e06bfad920e6141e63b310614a675666ee208e7b107747d7a57b0f52d03b5ecf", 232256, 0),
    "29798": ("100c4e5ef4a5754fd8a396fd91fa953703fbf6ea049d14ffa67212eec2079a84", 525169, 0),
    "29799": ("718923abc7b871ec6a3283107c9433315051f3871aa4b25c04478bd250e2305c", 47642, 0),
    "29800": ("3abbfd82dfc8dc604ce731bcb4edd93e676981bd13f0156a5b72df58c61ecca3", 181879, 0),
    "29801": ("c752d233318384092edd28eb738f8b53a4cf02877847d23d8f8a10d48e0d5fc8", 275995, 0),
    "29802": ("d04441eaa445a65be7c41b9300424b8b0e997424a8f1a2ad6784e55993c9ae1e", 158578, 0),
    "29803": ("9a49b8fc4b1cf9da61adce45cadc6c009fe1711906f5e65ea574031b5586917d", 201658, 0),
    "29804": ("e3b1536cc038ead8f04be37c5cf66eeafa86061ae1e8b05f600965ab32ef5444", 44618, 0),
    "29805": ("1ba19fc1161fa0643d394e7bdf98b574fe26c1a09ae0e6d76856ee5fb47e168e", 217746, 0),
    "29806": ("48760b1f9389f4b2b2b249abf76f748b8a21ff4481a68ad0c0b6861a0f33d379", 290530, 0),
    "29807": ("4bd808167d6c60b90ecf542efe9a62c4805259e606c7b959243fd54a57b5145b", 406162, 0),
    "29808": ("39171d5f029c46839b3b7d823a4acdc93d24170fed93ec77afbdea8ad9bef061", 377411, 0),
    "29809": ("b9839b6409a51d502ff51af56e467b6c782d41df94cc09be6d28a5872d797aa4", 232030, 0),
    "29810": ("f4d37575e2a94b37f18f2fb886d7ebfd545d18ebdc5bac75198d14e8bb562416", 311279, 0),
    "29811": ("3738f8d7b3480ac63a0f58799e2ade217b107fac31f86b242ccc3e1769c2bf50", 36891, 0),
    "29812": ("3d10acf022e0f2d31dec9058c29fbfe6ea27a32d43d69fc48b2aa2b5a8d5e12f", 344695, 0),
    "29813": ("c1c11c48d2dd6aa79dc22b6af2814e9dee43853de682d4cc0b59e913dade0ca9", 338234, 0),
    "29814": ("d6a59d15a9118b09db6e03727a068cdaab917b70ef1c9cc60f1950a7f8701909", 152910, 0),
    "29815": ("4b97dc8d91ba86407d26b6f0dc7bd744203d654f6f9fb0d21204abe357de3cb8", 363993, 0),
    "29816": ("d9a92bc5ae4876c0276bfa66858df931d2016753c22c369d9c617eadee3de400", 341561, 0),
    "29817": ("9a43c390d00d9456a25302b58924e3d9e561247ffcbd1493511625731dc84105", 45291, 0),
    "29818": ("a645d2d4cf8f14dddc8ac101087fa86d48eece80fd787bf489916153af526565", 66253, 0),
    "29819": ("8d84a9fd5e104bda8726dd9438b2bee542560471ede28d829b2ecf12e66f515c", 42837, 0),
    "29820": ("f7f0770e298746744e1a21651265c5cbb41bac1f48138ff6265b27162e495157", 70105, 0),
    "29821": ("cff1b37a175fd949f3335220b0dc3fa7f2ca349a161f48eb7cb8ac5205c82a31", 373794, 0),
    "29822": ("054fa8b6e3edd5441f6310a6fb1239f42a7a58c02eefcfdba5b639d9e52274ff", 300882, 0),
    "29823": ("cc037b4c8f0229735412f689becbcb5aaf8f364dbee0d9ee689bc597e7a90086", 157341, 0),
    "29824": ("c67f671153473749741612116e3face19345176d972fc62c5e6757c12fd6790c", 424572, 0),
    "29825": ("1bd5b519e8a29fba5a2475b65e37a62cf6b1c7910a4a224775e1175ebebf8a00", 34935, 0),
    "29826": ("9f6dfcdd509f2974dffbf152bcdb6104fe0095de2ef3a6f706f222007cb22673", 219081, 0),
    "29827": ("e54d8a9765a3a3191e8bd92fd5a2307993f6ff06cac731c8d264458567ed737b", 151335, 0),
    "29828": ("156754312c694379b1deef15f0403a60c8c28de6da21445d073fa92ba28da370", 135069, 0),
    "29829": ("ce98997aefae475337dac34196db3f9d2120dea9b78d953da4afe91feb7b05f4", 312500, 0),
    "29830": ("c140df5d30319a0c79f5c2eb6a07bb54499385e90a254e332189b243475ae548", 158682, 0),
    "29831": ("e0c3a6d74e62cb1fe9d44d9183f52e77dc48c9cf3258da8981895a326c9c4d68", 124324, 0),
    "29832": ("7dc3e23e9a6051058bf9cb5f3ccf30b8909028811727a89a8230b35695f6049a", 157619, 0),
    "29833": ("a9d972896c339964e8e6ae847b4f0f8ed92927104dc06c00fd52cce46a14c140", 314791, 0),
    "29834": ("bbda9f55ffecdab5e915a9ca121c8896afdd35a439dcb3b979379ba78ebb27ef", 112718, 0),
    "29835": ("0dc449131efa13e262f3f6e53e67a8408c78f4caba1b5ad13de3a3ed92e2979a", 650725, 0),
    "29836": ("6a72975ad1bc60225a0fa4ea2eadec0cd8e136625550d56d6d9c0205b73a7844", 56931, 0),
    "29837": ("c6a9c7f61bf6b08c3d124a9cdcb833f3e63764f4cc9c948b9dc92de3223d8f7c", 9447, 0),
    "29838": ("df029de24901a88df81a5d75a61638c978628fbbe0bdd52aaf795907af86175c", 351697, 0),
    "29839": ("72256894647c036c6edd98503efd475f736eca87274fc6e092f39d78e1ca7ca8", 278412, 0),
    "29840": ("063a7e05c87e9efebdfb31056553e7f8ccb2e3673cb6ad52bbdc5da38fbea535", 458885, 0),
    "29841": ("91d641b2899d01febc290a363b0838e8013ccc1eecc480756fe223a453f8bceb", 415795, 0),
    "29842": ("10a8c31503f56618f0e0f81792722b0cab77e815007a0cc23200e3559bd4e611", 235645, 0),
    "29843": ("b95db2d4e7f4633047b7dd86fcb59e5b0f898ad75a9395257b41797c7831123c", 365666, 0),
    "29844": ("d8d25e573eed3cba58725b7c473e5abdc44c0b09544cf5eb2a33bf724b0f9811", 41258, 0),
    "29845": ("78399ca0a32c0b0e7b66f9ad68848964d7336c9564d7b4aaa88b4d5d3e655625", 31828, 0),
    "29846": ("dd31f05d384e7379f4fa1c0f4a879076b5a25b924dc29836dcf486d2113be8bb", 296024, 0),
    "29847": ("cc8ca7ef3f31dae8782a03e8cffae4de07a5aaea8f5ce359c3a4bd2fc7e26a65", 20607, 0),
    "29848": ("63fccd507b10d898f7c4a1e2fa208c77aac87b8c97a6bcfdf66c041090283c69", 384334, 0),
    "29849": ("9057d56158295da1508c65889db2f5aded71bd40c5c5a94f44d92f072774253f", 203331, 0),
    "29850": ("8c620f36df7d85fa3451c743f1ae33376cfe413cc42e371abc9c95e515a89a90", 13873, 0),
    "29851": ("8da0593ca8d40e5a8323ae8b5b22947557aa12dec00d21552806cafed9d2f15b", 157156, 0),
    "29852": ("312027bd87f112ff69bc5fc4b3ed8640ec3a4c35ba27f9c06e16b54c367c950a", 293040, 0),
    "29853": ("cee16b0f4a24739f8886d93ce33da9b119a149e2f723535f639e65560d8f82e1", 30224, 0),
    "29854": ("e45dc7b39fe592e8fcc056577ae07883fdd291aae485ea61eed99d2378a91651", 202810, 0),
    "29855": ("ea863bf064573d9027395bf66a8823bd4b77850cc41e32a07995036d939f35f4", 155005, 0),
    "29856": ("ebe6fb96e634982d52e1de04ec7a892f650b32f9a898ebf4570a51670c7966cb", 321606, 0),
    "29857": ("39cad3a848dd8770b9f0b811f3fe6f44bea6e8a4dd3f49f11e23a47684f62c8a", 442161, 0),
    "29858": ("ac486c1dd789b23ce83893275d06c940420e49352cc1e0f94679efc45f1e2155", 44835, 0),
    "29859": ("e6ecef0f813ead568921e94c1ab0cae1d65b7ac741652b13da6ce352f9b2a65b", 360024, 0),
    "29860": ("cd1b20b16b1f1750a6094fb7a5295801f267146548f585c795ecdea9cf31d2c5", 112705, 0),
    "29861": ("9547e001963fc5f40799dc525b4ee735abe2c3bed26ff50b14fdd89999d25484", 350187, 0),
    "29862": ("4a95fe7afc5cbb350a2b7ab7aa54e893734e004193253dde6c68df941eff309a", 176608, 0),
    "29863": ("f13482011b51bac98c51333191dc49ca77eddeb57a06401a4c34e41bb29584a3", 21920, 0),
    "29864": ("75f3f8bcbf488a3e68638d2b2f7e42232d9b6938aa96361cdd95fe5ec6073fd3", 399486, 0),
    "29865": ("e0b2882dd26ddd86c2a4a8c1cdc4766979f5fb3528c83091cde8710809a2b260", 181975, 0),
    "29866": ("10b10a831107a71b081d23d0102470397b76fb7ec3451c9e45ffaa872ad20cb3", 387122, 0),
    "29867": ("e581aaddea9210301d461b6fd949296981f2dc2f235501d591ae8749976e9488", 36464, 0),
    "29868": ("519f3f44fd0b1072f358cfeb38c0b0094947b0af68d5bad90411dd0e092bba82", 470945, 0),
    "29869": ("5c27308a717e13f28d752e0a1f37053f7e4361854f4aeb05cebf7819b18b4e08", 38444, 0),
    "29870": ("bd5b2fcb7772118dd3dfb618cc7fb107ab4ea8147a2703aff6778372a276dfab", 255456, 0),
    "29871": ("b20f3dd88d003dda575298fefd59beab1ce7c0bd0169681c5b350df699ddd82a", 136816, 0),
    "29872": ("8854df3cc3d65677dddde8c34abb35be691bfb2cecaed8b4618ca353089e72cb", 26384, 0),
    "29873": ("08d6a427a414fa439976321eebc132893bef9f41fff477ac771f61aba6dd62d8", 343919, 0),
    "29874": ("20203cf3e30e598c5ad2715613bc430630b64789a1e7fdc5538c668a7bc99356", 26686, 0),
    "29875": ("ddce3e2f191ca4029f9d3751aa4cac699e0b906472582399e2594868c6c80a8b", 433027, 0),
    "29876": ("64e08f36a55fa80ac7556753bcd3f687861fdb4ca6951e81433501fa12935785", 146029, 0),
    "29877": ("eae555f3e43c9f2d291c329d0b9be7ada3105ebf8b3df4de2ed82ebdfdde6d8d", 363644, 0),
    "29878": ("2c412687c2ebddc26eac9f9d6ea026cd5a46ad1751e0545e5ca61f4b62904081", 158283, 0),
    "29879": ("f0ae8aacd8aed12f0650dbacfd940fee193eefc974d7e578023851df8e94ff2f", 320358, 0),
    "29880": ("ad69e5243f84affd6874579c4f4e40b96d81c78e3f7e257321b808bb49323a1b", 425143, 0),
    "29881": ("7d6c80fe0530cfc6fd22d9b1d02f60219208ff55f3b052fec9d116fe4493eea3", 355016, 0),
    "29882": ("d7806ec349d8c4032b4cdc768f65dfc47a0cb89bc9f099bebdf301f3df52b9ba", 217695, 0),
    "29883": ("0e54abb0080e25d7fd7b4ec403acfa7d801c96d37d1a45b1e329e2b0e8a77e09", 401414, 0),
    "29884": ("b247ba5a815df0e394d90c67454b8f066019272d64852e8d2d1807a5e806209c", 266379, 0),
    "29885": ("565796d60d5346a57b019f46473ade092378d9a39d968c49456587b721e1e5ae", 45163, 0),
    "29886": ("51e9732afaf804f16a96683010680b532a4660db16a1e30c6d900580739b12ef", 423880, 0),
    "29887": ("3b3685f5932f166e1b5aaf9609ff88d3c8289cae53b06f32463fd90813537333", 343056, 0),
    "29888": ("36ee713e2670edee4ce73d7321fc0bb9486c7754f80b99975515766a71bc4224", 176362, 0),
    "29889": ("1dc949bf65a6df8b790784cb519710a0aece208413875656bb972e4be8aa6f14", 486220, 0),
    "29890": ("3938eb523b39444c900c2d1e579a8f201e43b5e58ff315b6f87567b044ca0397", 256699, 0),
    "29891": ("6e285eea02e05516fad2b8b2416d21c3cfbb99b8a36071c04edff4cc5f01443c", 250683, 0),
    "29892": ("adfa33e978312772159d20f59992451705bac376662a5e82dd2f1ff096bf8b4a", 221409, 0),
    "29893": ("79ceae4e26eac4471eae61b153fa82c041c2b55e3cdb20de30bf8055fb5783c4", 30338, 0),
    "29894": ("80b60ab1b670764121eb7cfe9527b21d742d65fdd60ed8d0a3532708b4b6a411", 171884, 0),
    "29895": ("a6c0d677758a319fad37d82ff4e72ae7cd6c082ee3ce0f5f3c4e070904c87763", 329105, 0),
    "29896": ("c68f5844a7bb9eef2a58349d9aaf413d5e08a5c32bcbf07a6e571e2992201155", 54418, 0),
    "29897": ("7b84e61344bdac82853bd92b078f40a66aafef7feafd616d8aaf982b007d6a3e", 210137, 0),
    "29898": ("e5d9e2bfdc9b834048b27482465de42eaceafbca929ccd8737241ca04322035f", 47292, 0),
    "29899": ("f5daf4e2a9801bc398ab3969e45f2c06e17e4eca27773a233b81e4686af649c2", 193799, 0),
    "29900": ("d37a8cb22ca826b2f93065fdb6197417745b3e4d621f3d63d215c31d52d43b9b", 269812, 0),
    "29901": ("26ae57492fc829277d1c563592546613bb8654f2c316a2864c8995833ead5e9f", 354840, 0),
    "29902": ("d6b51939865c7d6447282d00fff885d64499ea45667e32ec728928ffd34593e6", 308796, 0),
    "29903": ("1001c263bd5f61fc77ba29f23fc5a6110302b915778570f0e56265d0d80bc4bc", 327179, 0),
    "29904": ("be8572affcc124958453fe2dd9563a8dddc3f26c7eefb835c7588edfc0ca4150", 350369, 0),
    "29905": ("e9a7c429b75feb89f00bcbc04890d7ab4ebb1c899fb529cb1bb0dd4fa2058276", 666604, 0),
    "29906": ("50bc6e01f65c13f1ba5be935dabb08b594ce0b7b0792b2da18c1ca58fd31f74c", 315611, 0),
    "29907": ("492557dcffe67f2f77bae127aab297b8fb1377ffe23479ecadfbaab0eff7e842", 50758, 0),
    "29908": ("d3f4ad2154d2e561d904b9b40e60d4170fa5e96ae82c0d17408d20a7cb4e18d8", 391628, 0),
    "29909": ("77cfe3c7ace2dbf5dbc11eb161851a623795d9c8863b812b3a55054bd3e84ca9", 33810, 0),
    "29910": ("773d24c3642af73fd594a94aea0a2279fafd7703e6158f8b57990921fa29a8a6", 143512, 0),
    "29911": ("35f622289ea354b103d06b32faf7f246b4dcda578a750863fd280f1c9ccc2ad0", 588105, 0),
    "29912": ("cda1b5197a0b4c4ae52e294f874d7d58b4134cb7212ae9c322c32e129e661d62", 95025, 0),
    "29913": ("489740b2b6591287ee9c59144611df6d6b6742d2b970c9a7ef02a74b374bf7bd", 369235, 0),
    "29914": ("2fb2b04c3509315246ae70492258bc64d9613623d3792f046e6aae8979625af6", 433789, 0),
    "29915": ("1fea659557baed69f07cba2635d02ebe76d2570a6e9f93aa51daa9d1fc1ad3a0", 356727, 0),
    "29916": ("fb7789e515f353d2de64fae91c73e809c57bc96e4078a8c5e120d384157ed55e", 320303, 0),
    "29917": ("289b1ce17b4ad8c960f1da809285bfc3034d82d674e3dbedda403e48272aac3a", 17564, 0),
    "29918": ("6725fb1e23e0e72cbf9bc450d0dd3cb175c3be1444fda811332bcc5309fe62fc", 408785, 0),
    "29919": ("894dd00881dd040fa05e5074277ee2c34ef55e9eda2d4ad1e903d0d92f6312af", 51338, 0),
    "29920": ("8f05a064c40d7c4d3b81dae0b0b21c9be66f09f035e8e96c5280e39aba9a2ddb", 340034, 0),
    "29921": ("be1b76828fb49acac70aa634ee09b59ab78a22a4f357df655aa19d5fd976cc84", 216071, 0),
    "29922": ("6d39c79bb7902a6dfebf79d080386e15c9af4be9bde68ed123b296d923af1309", 314600, 0),
    "29923": ("588cbf95b3306e3cad45bd1a933cf0f23b1e2d7090c4db2b73e788bff43c3d06", 228564, 0),
    "29924": ("7124eab0c351c49769c9aadbc371b5554b1baba046e175d5e5f877de7606d1fb", 193119, 0),
    "29925": ("c211e7abdc83cf9ddaabddb5905147620805b378d4429b70cef0d9398fe2d4e7", 394759, 0),
    "29926": ("76b3d18ab768faba062faf82e940fd69f844551eed2e8eb9db03cc4f98ba1ec6", 28284, 0),
    "29927": ("f9e9947c7ec95d28dcebcc29f80bee61463ef61f5ecf77de5037c8ef3eec7f42", 14048, 0),
    "29928": ("881864d6d1bde3ab2934d1ab8f66cac3d206d5229f9d7acd4fb4a7748fb69018", 215683, 0),
    "29929": ("607b254ff8a6b18152f7a13db476ed0a969ffca5c06c20f21c47b9f2d6c78640", 251318, 0),
    "29930": ("25b665a789ea24b1764b20e1c10bd37a86f5cb5b7ba9e31ab9c3493c003c2c8a", 255222, 0),
    "29931": ("a8cd3f2809c4a73180b9e182408e22cfd68ca66f0ad343f366458f3595a91d37", 88801, 0),
    "29932": ("a81e1534adbdcbda38a5b33998387d1d09d3780be37a521e80be6feb25a80de6", 178374, 0),
    "29933": ("a427850dc4e04ae512ac8f643c8836e839e61124c20bb47c18156fc332b27cad", 438440, 0),
    "29934": ("d50b65f99b28f2a37a875ec5bd2681c00a593d469c6cde01959b9c84acf5f7b4", 102557, 0),
    "29935": ("61e6a46b4196e4ca8d4025c498802eb07bcf2e3b13e65acc8444c301cb1b1b13", 271692, 0),
    "29936": ("5e1c02e8a715c4fab1316409550d402be74ef05e371bf2b3c75b8ad85f9ab4e7", 45024, 0),
    "29937": ("9c7ebeb9cba5790c1a90edb4f6fb713db68709c9fcfdfa934454bda0760744db", 88587, 0),
    "29938": ("513bcef293345224d54b67d964b7ae9b4227a4b1e06dc2d916b0144103bff839", 19210, 0),
    "29939": ("613d13d83aa482adda4e812c2cf68234d2f3ff749589bc50a32603f181fafa7b", 349382, 0),
    "29940": ("fc4c8864e9f88b01206fc2b83b1eeb6ba4c60a2146d5c26c1caf1bb8c85f54b1", 294794, 0),
    "29941": ("b301240f69917690ab11066031fafe6021e9c837b7bd9bad913dadb97e239cf8", 428491, 0),
    "29942": ("d4fa60eae71db77cde7e1adefb908ff26dd2ef4b64f840d5f6cd2d17ad3ff19e", 472554, 0),
    "29943": ("e83a270164e3f37761adf3bc12362e9d1d5a8a4389f666233a967f1b28a8112a", 509787, 0),
    "29944": ("92f6e9353634b2eb411affb00445dd4c7bbdbe3d83eb40fe777c5d387dfa9d31", 234212, 0),
    "29945": ("ce237c3bec5cc7cf2bc16fbdcaa9b35e35267f350698c338fee6dcaa0a3374ed", 144498, 0),
    "29946": ("f41d3c5b282d309860be51d9dc6f69c26eaae73bb7c042428b006e8fde8f536a", 213113, 0),
    "29947": ("1c2725952f9119319b19403a3208a409925d97e077caae3fa6e4856ad2360280", 189683, 0),
    "29948": ("83ec0cda3a11e39eb7ac9b326c5f5315e22c246417d7767ab9dfb49d71c19934", 329393, 0),
    "29949": ("201e9f182576d6050a29a8a3a7604a4c8075b38d1dcd288221284cdce073fe47", 250680, 0),
    "29950": ("d6acd6861639dfda3e06c85ad0cd8991548d18216ca8f062b425c2f64962a701", 215597, 0),
    "29951": ("ba278857bebb0431a3423a63e77430ae6df6e13eab15fef61edbd06152a5c64b", 162039, 0),
    "29952": ("44e808a799fbd557410d8abe27d5fc09a3fb21b80313567ce5769c95d027355a", 169532, 0),
    "29953": ("f8895274b0ce60e90dc0ef04e43d343376a25bd37690812c1af2127ed5b3dd58", 180269, 0),
    "29954": ("657c4225719cc8c4114db0629c7e572daa533e9081934a697fd6461a7630f1ec", 354907, 0),
    "29955": ("7ec84d11b3ff20f52cb0e67bfc1e95961e06a673614e641b37425ec1945900d4", 238628, 0),
    "29956": ("340f80df2b31c41399f65ec4a63fe352942604c61472537b2018e296e5d513f1", 59146, 0),
    "29957": ("3c3eaae89011a58996420067f8e9d54ef3add5953e02b7946fc853281d849c79", 38775, 0),
    "29958": ("0631ba0a1efca02b1ab476450e9fe8a9694a56d7c15683a9e60d11609821782b", 625637, 0),
    "29959": ("72cc08bf673ebf2ed661130114b9c68f2e118bb9f09119415288a383166a4e76", 250745, 0),
    "29960": ("5e70073f05770966965359cf8ccd0003422734e5f81285401d75539d52e52644", 297808, 0),
    "29961": ("90f12d2e00663cfe64f1d2c8a74b991631eb21867e7c2c743111ba139b154530", 254933, 0),
    "29962": ("6e6e0ea8a9a0c07cda332e0504bda15681305f152df65f3027edca7793202685", 283494, 0),
    "29963": ("6ba4719e1b0c9a9b1ea5445c9864ab7de000f58346e42e200a2caf70b232c2d6", 175496, 0),
    "29964": ("27954bbf156be5dc3c555484c4baa405f31a4b5b57ed33a7ecce330f33a060c0", 39077, 0),
    "29965": ("857520da4ed5efa089022beee3bf9c606185d2a8f332efb681344f26930ba035", 11193, 0),
    "29966": ("dcfc5bed185a039c4c1d5a2df4bab33aa671cf147f6c31c209be737432b886e0", 489420, 0),
    "29967": ("fd7016e1d22c5bac791b18d51c0a285e7c75adb90ffdb679e456cc787bb7ee4d", 159365, 1),
    "29968": ("523bc685308b8f5819c28a3ae44be7d8384a543b240d25fd020b8a4f8f37f63e", 33888, 1),
    "29969": ("51fdd80d3fe3487d0f01b2b7179b2a1732c88c142f641fdd8f7055800dfbc043", 149236, 1),
    "29970": ("3d224cc695fbc3644eaba6fb8dc93827a720cd5220ebfebdf1d3e8946dda6e30", 192909, 1),
    "29971": ("59e69fab6fa0eb1f36ccb78ddf572038462ccb5c6ae1f2bce7877733645b60c3", 362535, 1),
    "29972": ("b28da0687d8085c442539443debe6ef67c81550ecb7ef443ab8c28c644ac5041", 262250, 1),
    "29973": ("e0d6475678c09e836b79e78911197244b9b89ee2817ceb2686a0f75065f92ff3", 138252, 1),
    "29974": ("fac75e807971a89969a5db5b0aeae62dd89dec683b80dc9be36d04b20771fda4", 172818, 1),
    "29975": ("b9f08527fb194d5e22bd59b97ba21f02d6383401cd09f133e7f128ec75a1c036", 301226, 1),
    "29976": ("809518a2d4e7642f4a8f6ad42a92e8468397204ccd0905b14a4573feddb8ae6f", 324405, 1),
    "29977": ("ddb327b43c44662ba13fad6a7d3bf6dbee1eaf2404dcaf0386265c15a61bd34b", 298427, 1),
    "29978": ("d1f03b88034644e32e20442dab968c85b011165e5180fd721723df368796a375", 268132, 1),
    "29979": ("124431b93b9b7aea95a5bc4069b7a0165a48440f507bef3fd1bee279ebbaa99e", 400337, 1),
    "29980": ("72501106ee0aed5db699ce92aba9b51b852a0a6190e569b76b7666118669e62e", 223038, 1),
    "29981": ("02281fa5a21ca6b70d971ef20e7c404330eb34cfe0f1a26e6074cefb650796d7", 63099, 1),
    "29982": ("3c09e99486e9feafb10f1578bac47b1b5c3121d3081a438a1d3b6eba754106b3", 229572, 1),
    "29983": ("a3005603001071be73394e57bd63d4dacffa28ae22ef3ccb41bea8ecb97f0759", 466513, 1),
    "29984": ("2794d64c8af891b394368bb43ec1ab57bd27e637389413150f5fb234c407ca19", 439638, 1),
    "29985": ("6d9345b61f4e6c7f1e283be7c25e29afb59d08b43d855be7181908706c2b6b81", 323879, 1),
    "29986": ("48db27f839e6782ba42331de48837b51a7c530a20559671890797e3807f07f52", 238208, 1),
    "29987": ("4742d85e183b88a191e31680657363c0ae0593aee04586568b8602b391950fc7", 392046, 1),
    "29988": ("9e023de23f0c7d597c0ba432cf977f3a0d3287ca5b28cd2832fbdd1da7076ea7", 235315, 1),
    "29989": ("f219807497a4fa8c669b8eb60439a8630628e5e14196fd2f0561fecb161e8e9d", 5646, 1),
    "29990": ("fe9d6af56d880a0edc9d9893471c5f02742336bce00e8b468ed67671e0663e98", 173346, 1),
    "29991": ("fb48b6827da6b5e0a2581771b93773ddccbe52a8302b4c365f1e5dec33c4fe8b", 322851, 1),
    "29992": ("5c57adf7cda2dfea9ac82e67c00a70a02ed16d89c2cb3731eb18982738f6ccc2", 411612, 1),
    "29993": ("628a1beae9da57549e624b0cb000db59c0a9472b5da19964a207f198b7a18996", 53392, 1),
    "29994": ("62cfaed11c1c47f2ae56471958bcdccd54227974506db8012e0cc9b11969343e", 409145, 1),
    "29995": ("2e87669eeef36597bd648b380d6a1c47765f0e76942de5a45f0c34c818e747aa", 341884, 1),
    "29996": ("0aefa71e77cb53f6be273b65a4e24047f41bbea0344f7311383a39a62584fa79", 278785, 1),
    "29997": ("2f679d9282b5899d5f479d3c99dc3d653699edc1a83e7b4a64c49e54a3cf6614", 283252, 1),
    "29998": ("5800e3f5fe82c5295e305afa9d59d207cc66d4d15010b26ff16f55af915b7551", 184268, 1),
    "29999": ("f8a071d1036a776b2f30a065214381530e0be9a546540b3645613948b17095ce", 546934, 1),
    "30000": ("b2a1756b511bf4fd4f5f8f84b659b6555e128f1c458beb92e5994659cf92ea91", 423591, 1),
    "30001": ("c5a6dcfad52f59182ac5c0211a0eae6de1b738cf40bd91f53fedb15b57e43ffb", 281859, 1),
    "30002": ("0491a998ce8454b296ccc260e005ebaa50a7bad109a3a71316d6ace20748e787", 12043, 1),
    "30003": ("8ccb9d725d23109644d5c9bb666ef94ddaa8f98cc52542c97ea4e71991ce3914", 160924, 1),
    "30004": ("d262be4e64938f3c00cb39c36ef417bb463c35aa35f8ccacf94160d1e7e07c91", 320305, 1),
    "30005": ("ac68c7ba0ce669fbc80f078566ba823399b5d6f02d5ae532ef62a421131457e0", 35902, 1),
    "30006": ("1fb6cb147b3b03b26535df215f3df4e5fb5dfea8f65ec5ef391ba3f92cf28713", 372456, 1),
    "30007": ("b39a8c536b4802a6dd8e0fa9d8eb78485efdee496a6ac8b0c31afa4d6f8d80c4", 431144, 1),
    "30008": ("8475d13328ea4f75463aa994e8c600d01a9b59b0edfc6e9d4f345cd644dfaaf9", 84409, 1),
    "30009": ("5b7dac0847fd4d298ba76ba974373e8da2efccec8bd1a7e8b659dded68102080", 317548, 1),
    "30010": ("a2374f9c1e51d01adf7a923a0476ea649b9544eacd3d2cc63d84c8a1e29100ab", 280006, 1),
    "30011": ("e828c7c7735b355c605e61919b45eac264e8ed401ba1da9c9faa848fec8f2374", 181483, 1),
    "30012": ("5a14b5561b5132bd7ed187a4754326a599b3a8e227fe1249e6feffe85dbe2842", 296826, 1),
    "30013": ("00bca3a6fd6288939b578d505a59a91fdb3b72305994acb5542ec7f210403068", 84731, 1),
    "30014": ("816981719a4f5e1e612804f1ed4340dc79692baf5cb28728be847e7f1c298c7a", 25763, 1),
    "30015": ("551c5a55b4d4eb297fd5640d15ad5d1a9806e45769af73805f6c7933a4e58cc2", 492728, 1),
    "30016": ("f35c47941395010fe96bd7082d1fb36b355e0d959209822033f1893a851034d4", 28128, 1),
    "30017": ("bd7743b7b5cd187a9df03dbe9a1d82d8e07a69e7e8f241e62cabb0e24e51204d", 169696, 1),
    "30018": ("96ca4a1a38c89276660d3632b568b369e36ec0f9ccb553c565b46af95dc9dfca", 292728, 1),
    "30019": ("ab1beef8bad6756e7ba284ce015333cb1f052fdef9218abb7320cfebf5db2512", 274720, 1),
    "30020": ("61ffddc264c306aad1498e364b88a3489ad80c7f4dd14ba64a1873f4aeb46e61", 40331, 1),
    "30021": ("f58b395da4be82ca2bf7c36fadbf8fad3353c07eb8c4e971ad7a851e5a27836a", 274123, 1),
    "30022": ("5eae0f649a1079417dac0ed2bee42bf1e1f6c966016e8eec2351e65ed46e3ef0", 112976, 1),
    "30023": ("b9371c97eaff4ee86ab9324943c2a59abe630ec966fbc03d8783fc20672d22a8", 55461, 1),
    "30024": ("3988dc07a8ea24e362e27451a6d3bc8f302cdd58fbcad48d847a72e934d6e55d", 274752, 1),
    "30025": ("3b6d4a1488b8ca62d364c6872d03f24ade0eb45797757016dab63ef60f1d87e0", 391432, 1),
    "30026": ("11899c0296cb71605e19a2f286da4fce08fd7e6be826ed97e003145bd8bce8ae", 446383, 1),
    "30027": ("a2c8fcc54dd38229ce269cf9928dfedeb8af6ddb77384f864c4b849fef24037a", 77722, 1),
    "30028": ("0357b1bc0fc8c76dd894a6b3a4b74dceb302cd3ce14ac496da8cb06a11da6209", 232847, 1),
    "30029": ("6d82c2a777e2e9e23f31361131bfecbb85b997fa13b6226b93bb91a327ecf31b", 268139, 1),
    "30030": ("2119dc4b26825ab3a95cf0782fc140288d7a3a4b52ed29f8e57718c0a0759285", 238033, 1),
    "30031": ("46b2dc015d04e31dd65c8ed03b3d3fe15d65a2634006bf00c61baf409ce0a354", 221503, 1),
    "30032": ("d01d7a1dd4f149a416051d712c52d5e4cf93e9a7a92acd8a2ebe3aa94693bcc7", 459675, 1),
    "30033": ("9cc74da43d6d84b5480362e7f8264d0ef222140701ce218d92da6fb2a7b3bd75", 224771, 1),
    "30034": ("e815d3f4b92c44c9e3c8ec56a3bad959ada453dd211a08d445b5454016b99615", 275577, 1),
    "30035": ("7e761b87f20505e6c6ed2013b5931823e897f5f3a53c43f9c3f87b8ca9a1b1d3", 36134, 1),
    "30036": ("eb14ea15d70a2ac29c699840581627b7edb6f73937fa4cff8688b251161c6cd4", 92336, 1),
    "30037": ("bf9eddacc0bb6146ed0b1aa396645bc3a4accb26d61d41a403beab9830ea8883", 207786, 1),
    "30038": ("cf1976a3fe3db5b0ddcbf5f6bc15c8d208a9989d3ddc39b97cfa5894547479db", 139675, 1),
    "30039": ("8fd38d858d925be76bf85fa9e1649f47c754e23af8efcacbfdfb6cb2fbf4fbcd", 197925, 1),
    "30040": ("5e29599774666e3febdd63e9b2596f98bf79d4f993101c52dbfad52527889d60", 314066, 1),
    "30041": ("cda30a8f10329559e3a3a56511ff578d356780c43c40bfc86c577b9bfc447115", 299860, 1),
    "30042": ("d0e4a2632461c4b83a4c9c676888c3cb8745c9a637b5506a7ef91e4a0f9f0e44", 454800, 1),
    "30043": ("def142592510611549426e21dd8440212ab06cc5cc6ec1aeb8a84183753ac547", 328862, 1),
    "30044": ("17f214f4acf8351ae8826c1fda091dcbf9642cc3b3339dd366c465de87952dc8", 313888, 1),
    "30045": ("47614ce5809da7ed290763091c19df53c59b05d9ab3eba9120477c0707d9e25b", 338716, 1),
    "30046": ("be18a3895ed6b89848c82adb18c3792916e13fc8a750e62423d1d145de302ef5", 138255, 1),
    "30047": ("ed6a6a4328d59ec1759fa447860d67c7b15cf12d57d44cf53cf0c1443eb08560", 451212, 1),
    "30048": ("166d7cf8f2891bb16cace519c948935071c136b22d5c01826b90f329f9545ba5", 190136, 1),
    "30049": ("ddc75809e6510d411fd478609c41f6bfe4bb2a1fd3f06bf76d492a8b66b43662", 441054, 1),
    "30050": ("bb3c3b793a3b95986433331bb688f4b7f9991fe987714844f9713fbc1895de1b", 44047, 1),
    "30051": ("67346bce92416848e2aaa1c4fd8f0b8f6f711f4a0edf8eea75b281f9ed1f3386", 346169, 1),
    "30052": ("16d49bc123bbafe851a9ccb65a22fe6c60763c85fc8c449242cb1614d0ae386e", 39776, 1),
    "30053": ("6b5badec65cfdf0b78c03d845e5af092ba528a3d993b3ed6c1012d15ef4a7050", 202646, 1),
    "30054": ("14dfe138cd09eb23efec9b22ad17fa634c27e014869f05ec3e5904ffc20320f1", 127404, 1),
    "30055": ("648b7704ee679bc1665c619411d283be0e9df343a6bcdadf2216c4fd74f865b5", 314128, 1),
    "30056": ("29907c6a6302d799af61050032d7fdd430f684dbeedfde8c8c4290e605ea012e", 235175, 1),
    "30057": ("8509f921dca78753c16a0404eaa13d2c69ba9507d7a77dd26ce9e34d09b7be55", 154631, 1),
    "30058": ("7364583b37789317a6a4fc7e29e5fa4c474c05d3d400c5d89f94304230456a18", 27146, 1),
    "30059": ("2fc8d1e55becaed577addb001d1f73f95ba5d2ecd880d36ebebde4d8d53cb82c", 155987, 1),
    "30060": ("f0dc6f972ba295330efd37d914b3c4a2ddb5fa1cf73eea2441f1f895480c97dd", 332133, 1),
    "30061": ("a5e8f979b85edcc4f8f7d89d7dda8dd7465592dee5cc59eb505e569933e0b405", 311476, 1),
    "30062": ("324bd0fb3797fbc747de854737b55a0dd0cbaca5a11576141c380a733ffc6e85", 418311, 1),
    "30063": ("ef2642d7284198670f5b3ab0f4dff4013102c4943a9810c610757fa17354061e", 177610, 1),
    "30064": ("9fac1cbe03d51f5b0ad9107aa9a3f28e1ec4616fac1a606f225ebc5873705f8a", 169072, 1),
    "30065": ("59114cde46c52ae3427988eb386e0530ac68a353976751e6de8819c3f14d4865", 382422, 1),
    "30066": ("f0164873b4b2a818071c5445e17a8f432985456a94616cdf1cfe1737e1324d69", 31233, 1),
    "30067": ("9881d25658025b02a641361db014ca694566259c95935260ba084eb506c6b13f", 287497, 1),
    "30068": ("1b2f9cb2206e716876aef937a2bd080c87e60aeb1a54dc001e5bfcc8b9f3a084", 30526, 1),
    "30069": ("46afd1271fa651bde5680a151a3783cde12014947f0a56aa75e4d68562f69618", 36387, 1),
    "30070": ("01074474532cbfbfcbfc1a51238d497f3155205a8ca3110ea32f8744f4a78522", 278402, 1),
    "30071": ("acaf1dd1f0a42c09b00d7bfa4fc6b86bb1742dca3dd2bac61f22311763066722", 464357, 1),
    "30072": ("84aff4a939d8f175a606ac8a82230e5f63a69f26e393bf9efc17bf92489047f4", 172727, 1),
    "30073": ("f91334cc62a9b53113381da9aa5834f35cc31b70cd4a25ed9e03d5b2a8834f9c", 178167, 1),
    "30074": ("95a2575eff65cdc5e7ea4475451ac553c28f37b8ac79bae3b197807b8857a9b3", 389772, 1),
    "30075": ("634aad28b6c4d343359697c562e5e261abb4ace0d5e29ea1922c1a60ee0ab3e4", 223267, 1),
    "30076": ("1eceafad9d42b6b6af8aeb9a6665698c2a2cfcbbe7f5e20a65361ae66fd50f41", 357502, 1),
    "30077": ("6d1dd6dec9f8adbd42401da742d6eae091c29f983b4a5102aa6dbcd174f4b7d9", 33011, 1),
    "30078": ("16244bee7012696e835998b69b0c2f802cc93dd79e45675e49101272010a525e", 279513, 1),
    "30079": ("778d24e97e0dfc1bf33b1294bd99186645af7be24bd978421abdc3ca254c4cda", 163570, 1),
    "30080": ("f39ef21707c058478ed4146b513f669eb57a54bacf4d8f2c507f1a895c3aab60", 42325, 1),
    "30081": ("c54663ba9c7026cb197c2670f861a36311a6feab204fe097ccd4337f3ef6b70e", 142497, 1),
    "30082": ("d17aa6a5231586d978c3d446aaeb1f150bcca4e108e359897aac726fdba59831", 231909, 1),
    "30083": ("011d0dece778eaeafe0c0a627a844f4a99e22f2ff116a4626f5fef0c23628c03", 157569, 1),
    "30084": ("2eaebdce64b96a739dab39b27accaab46f09ab9907d54f38c216ef17c5aa7c97", 430091, 1),
    "30085": ("7777f9d3392af19ed729f031d5a7ffd883bd7e99ba0e05217c5eecba2f1cacd1", 403083, 1),
    "30086": ("a31d96116e7cd0901c2823111d94741400422e2f51e97a28b1ed9760f5e53e21", 151815, 1),
    "30087": ("124fdf328e4390c9375b35355ef22d5daa2adb7a0c6cee0dd678002161058881", 22300, 1),
    "30088": ("888209e6300d4e94539b9ea911dccc44ee91a24edecceebc6e1ad55dca55d492", 281225, 1),
    "30089": ("062205e25e353ce74c8e50688acdb6fd365482089ac5617ae0d96b409d6012bb", 234545, 1),
    "30090": ("30e1125a7766338545b2cd2eb72e9bbda73c4350b725ec89654c3704213f8963", 535379, 1),
    "30091": ("71a9fd8f21751564a26c11d531ba33cef789c2d9a7d58b42919f47367f0c679e", 186800, 1),
    "30092": ("01c2fad6b27dc7ae49d50bcb168999060a5a4215d0d5086ffb21dffbe9f711f4", 363075, 1),
    "30093": ("f39c27db1fb14e23626a813c58664d8819332b13fbe7681c9e01086279fbfaf2", 131315, 1),
    "30094": ("f09faadf9ab0d97fb672f31bf4df2fb2d06d411b003a5a8c89f71e122e4b6142", 122312, 1),
    "30095": ("887a0648600392bb8421048fc1c542cdeed770dc81d4b70f0305bc76089d8956", 463088, 1),
    "30096": ("fab2dfd8b7ad6fb2b2c85e16df287998f045ce0066b6931fb7f582af4d6b739d", 441662, 1),
    "30097": ("aacb7d27d8fc28e85ba2492a5101991d289a082c7c8556d235d5f81e7fe6ee05", 213288, 1),
    "30098": ("7b181d0f37af0a4cbea86df9dfc4433be7bb3a9e8c75c02c18e59b79d511a2c2", 149987, 1),
    "30099": ("a9a44a988fd89e5342cdfb8c2d523b548ccfd5cdeb5f781783d093b80ab2d311", 201581, 1),
    "30100": ("ed22a923e3ceddd51d09a6a1a3f625b5535485cc5d99349a4e4dee17d1bef399", 261978, 1),
    "30101": ("976eb28ad53854665ac93428c5b8fe210dac55c573c8b8dbec968ad975400cb3", 178449, 1),
    "30102": ("e12c3679aee1ed9db5815bf35f9e41dc3243dd3057619f4bb57f61ce7979c052", 118565, 1),
    "30103": ("a11cd5384154b074b2a2bccdae7abc494139c68e3992582715ba075a641c1c27", 175261, 1),
    "30104": ("0c23fb5e072053bc05b491194a603b24502f1961224e2acc87321e8a2bc477d6", 258195, 1),
    "30105": ("9936da1af75ace8176aac8ef68773a6edaaed0033ae1324e2d8ec34b4094a1aa", 360428, 1),
    "30106": ("c5f70f651da1bb27e2f46c58e5993c1236ea371ab80e456c4ac2b29963f4cb5e", 65747, 1),
    "30107": ("623c694a44edf387b7aa9b6c071b62ba340d5df114b84c88d22ba43a53ea2aa9", 17828, 1),
    "30108": ("45f417a1e13ce83b149c51843d89f515df351646cee496edc414f8fc3e48a026", 115395, 1),
    "30109": ("0756bdba68968ea9104fe3c70dd901d9a052e7961e414dcbbf1df150d0ae704e", 442383, 1),
    "30110": ("2f37576b77229f325dd6beee206aa5d65e53ddda0dbfd0415eaa91167822996a", 345901, 1),
    "30111": ("84a0d8aa01dcd7ea1ff89995d625ee5de52bed39c6c2b81830ac14dcc4808fe0", 129471, 1),
    "30112": ("24d8a632f03c58358a76f85e79bbb50a45e56728ee29f8fa6d82449244adfa4e", 141709, 1),
    "30113": ("76784b9093e44ff9447ff60a6e8957ee69571c85c237344962bcb9890438cd96", 91526, 1),
    "30114": ("f563aab2cb7b28f398a4c409a59df6207e7e16be3c09e7896bc3b588b1489bb8", 159718, 1),
    "30115": ("faa2a03253f49309ad2c3c7d34018f59126937bf1fc7b22972be01ed0b044e90", 27649, 1),
    "30116": ("3e3ae31eec3c28a18a7edb524a5271966329bbcf03bf7ac6d6fd301ef73a4026", 179872, 1),
    "30117": ("91c6cd7180ed8e2b455eff9356475e123b7db5a20c9300a82d457ab29fbbf7c7", 201708, 1),
    "30118": ("86d20248abd56ad53013de26302c605f5f3e59fc97dbf38767c2b6de90476346", 359295, 1),
    "30119": ("acb9599a916252f541645185aea9fa9af14232a9d2abaf34a21fb4f1d8752a1c", 342010, 1),
    "30120": ("1f3da3f863ea9a60a5f35c0b3c2a5a6ff57847b9e09266ce584b90a9cd108681", 141543, 1),
    "30121": ("756e955290c42e34a3372c062d2f6d673ab1069da5391aabd38fc2f54202bbd8", 484641, 1),
    "30122": ("1416d3105af65f144983fdb03c131f3e0a1ad9286aaf89d14e17a1a4c0f56bc3", 230374, 1),
    "30123": ("75b1d971943d32c22ac05e0a3ad4d1382042835714a8ef8c994889d6b4dfbcde", 360035, 1),
    "30124": ("f87bd8fc4491a24656be221430ebc271ff31157bf0be637c16f82168353cf0fc", 262405, 1),
    "30125": ("20bb98b31c73d92aaa0af91cf9a1afcdf91759095b02e7bdc3dbea0bfe6e9706", 165844, 1),
    "30126": ("0823073bdb3843bb2bd6ae0b3a078ee098a3954b137081b14e33989368aafc9c", 317604, 1),
    "30127": ("639953f7e474cd2f5560450099701c881a299a36ca18a20e6c8b4b2ad5061a4d", 303591, 1),
    "30128": ("17e5e1de9aad1416c4edee5390887bf556255fe601312de089a4a9cb0c07099c", 250436, 1),
    "30129": ("16050e8cd9d1c7d155a56e7c4c108cd26755678665a0ae4767ac00a7dc54512c", 195967, 1),
    "30130": ("a3198b89ce74bbe83e791547ace63618dfa1cf3e444d43905810cdd7e133a35f", 35551, 1),
    "30131": ("2fbde6204fe4ad0d5b73ba9448d6fad74a6250566aac2fd8c68b71ffe5bf2840", 210231, 1),
    "30132": ("6f325e54e76e94dcdd3644f6506f1081eb158af1d7e34b3d5fd75e3547dc9078", 177593, 1),
    "30133": ("657bc6eb4d1286b708020b8b1b2f6b740b6d15acd07a432855a5ac5a6d789021", 158314, 1),
    "30134": ("58afe2db21f3a8a2ce14d695161e3bdbee11a9dc97daaa8f7e03dbd14c1f8e98", 419450, 1),
    "30135": ("19688519bfe16bed0de1caafee375af627915c54ae6d149ddfe7813f02798707", 32626, 1),
    "30136": ("d49048485e114b93d306c739085c55fe1baaa677ffad38c8e6944a15006feb85", 38629, 1),
    "30137": ("a06f62d3562aec9cf4ff0ed202f50c2a095843e602f5c373acb56530b6b3595f", 243732, 1),
    "30138": ("c0c382ec6f67b444ec8d513f12b59919e23ae1b2699625ad0091a4ddce71d535", 47939, 1),
    "30139": ("19b362a29ba15b2a7458c3fa1167743e7e2c46494c2b787f3e40c691e8ce610c", 311500, 1),
    "30140": ("ef367c27c95c90e291b7ed9580e91c0b4e0668a1b70729d75fe538c805b74968", 79826, 1),
    "30141": ("bc93293afc705c7d961faddd33953b2be43d39c58141bbc9cd9165ebf11537ed", 282079, 1),
    "30142": ("24ac8654092a69f166fd88dedd9fd7a9e8cef128dd8989b19a5881510cf4c495", 125477, 1),
    "30143": ("72684bea1309bf712344bcf060705db8914e20fe47e1ebf4c9e72939c2979015", 144787, 1),
    "30144": ("5ad7360826c6bc88a0a4d492e977daa514234b96784b27c310bcb2ea99ade41a", 295378, 1),
    "30145": ("1e0b1aed58268903c90a92addc0955ae3099306a0e6453742f55e6e22f9c96dc", 429015, 1),
    "30146": ("61788979bbf1db22d686aeb2b32deb4cfac9b7db00ce669bf31662a6ab42cbaa", 335586, 1),
    "30147": ("c2b5d3cca1ef10b277a969d095aeed9be42d4e33d0f1fca841be933f1c744941", 139514, 1),
    "30148": ("a5792a1da7a88f1f457df807ad1d24c833af76cda040390ba8630ad177a32057", 313057, 1),
    "30149": ("ce0c17ec724b77dec48982282ef677b213604142bd0bd37f0ccf4e0f5376bec1", 133066, 1),
    "30150": ("0e0552282675419241bee101daf467b7154fcdc701adb90423ed201e0321448c", 308194, 1),
    "30151": ("e5c71af88089a415de835239c4122102732217d2286c3721331ae1c8159fa87e", 218048, 1),
    "30152": ("040574bbc5fab49f72d9e4fda55fdae27efc0a1c129b8bda00837a97d404db0a", 585431, 1),
    "30153": ("fdc43339c2fd2daf99cfcb87993a170bcb6cb4019522a1ee6358dc2da53c0da0", 121490, 1),
    "30154": ("2c08aa1d277946fb6fc4bd15a824e291c2b8bdc81391b5a76d069af2562a793f", 36086, 1),
    "30155": ("1b3ed09f7f9e2552fdfa4794cdf5c68bef71a3a896cf3b498000efd457e37d7e", 31791, 1),
    "30156": ("6e37c824f81561bc74a87770ab5659798c68331d550d308b7b05fe087c44b350", 269023, 1),
    "30157": ("a804f175b5d28e3ee53036f730651b20b75ddedc1ae4f3e5ac2d54f7c7503b4e", 176878, 1),
    "30158": ("f1d223f10fff4d00bfa405292c779b70e1200e8eba9e97fc96d4d772f8c12af6", 321800, 1),
    "30159": ("d7016b0a538caa61924f9d98e9706d3aa0635c84efee6b96a2fbfc1b53409f9b", 434150, 1),
    "30160": ("9c2d6a7f7cc5627c5f76782afe05ff8a065df7408464c87859597b29e24a1d7d", 36291, 1),
    "30161": ("70910833e8b3da496e20d89f321cf16e615c361086f69a447168e277e1af201c", 313454, 1),
    "30162": ("3f82b7b23ee847a42fb10ba89df3cf72e7ebc3dff4898a66020b6d567df5de35", 191575, 1),
    "30163": ("2b3921233d81363df112b2db41898cb0a223fdcf2c128e2b61484ce60461a9da", 295826, 1),
    "30164": ("370eb5ff7198c80b0a6d4e95d24b9798bf7f82824e309c2b2b987acc1a535164", 329710, 1),
    "30165": ("f40bf651845a0397c3eb260395392ce02626e792c924dff582d790a50f51b22e", 205311, 1),
    "30166": ("72d4e4c636531b242f8bd2ade21b8500bc49b34e0aa561577911aaf6f042ff39", 302420, 1),
    "30167": ("6c32a7077431f1ce690535aa08abda7ff54963e94a9fc0e13da7889394fa387b", 35317, 1),
    "30168": ("b3041d53e17e5050dbd4d9990c80aba5484e6b388bb5672285158d4de594e8d6", 230393, 1),
    "30169": ("b9a4df642650d7c21476718b79677bf149517d09f1c1a1a9e1d56bcf71e8f78d", 285614, 1),
    "30170": ("0abb6ffd31076d52cf53b23e1684e0d778d0816ff749e3a3410cab0007bfe58e", 339852, 1),
    "30171": ("6002f5b926f0611e7cfb81b44802ee02a1ad34edf600771dd231d50e4d27ca64", 49829, 1),
    "30172": ("dc0ec9841c3b6715b5eea60a4e5b835111387069227d1618c12ddcd4d5939776", 267752, 1),
    "30173": ("2ef7ca66f86de2cec34fcc72a35c020eb4f4bc2716a8770e04dbcf1d3553f646", 380333, 1),
    "30174": ("f8309771eea09d6b82523611354b42ae5717204f627a668d64e8e62a9ff9145f", 267044, 1),
    "30175": ("725756f3c8635192d14a279c2f565d17f2019272cd80244f29b9e50494a8546d", 328145, 1),
    "30176": ("daa7eb87cfb32e775b44b552e95c7b3d16c127fc31315fbaa2e298e6cc46d61f", 324575, 1),
    "30177": ("47fb8b33fa74bc7be128c5bea16e5524b4f6dd154c9994ccf93ea974d658e601", 481995, 1),
    "30178": ("33923999cbc72b75d609673d6650466148ad46577cdd41afa50958ac66ba8827", 443951, 1),
    "30179": ("6e9c5c70580e15f9228c8c5ff6861e500ad5c0b9b05e3a6f85606444498861f8", 35468, 1),
    "30180": ("3a03dda5c34d3550642bce83a9c8ff21af783aa2b3613630091b6cdee80d530e", 32473, 1),
    "30181": ("e9fb61075c26f38b069c098e3399f68610468d75c2bdb9b5d00723ef42a93931", 284044, 1),
    "30182": ("96dd218b3d882c17042c9260815fce59aa44e2498eacfb7f9028965ee50bb2a6", 331371, 1),
    "30183": ("94310fb117acc2f17f668417a267db60db36d118d13ae7a60c23e329e973bec0", 296928, 1),
    "30184": ("a17ff99fd6727220157f2fd95c140cd453fc7a8d757c819419f5e223c8f68a6a", 323906, 1),
    "30185": ("873fb550939e186a1ea2bf24865b49469ecf6dca5d51d6d71d7852f417741489", 275412, 1),
    "30186": ("ac1f9a412c447f3bd5c63712525be04930cd744fbf50abee87339c1089999ba4", 399112, 1),
    "30187": ("9a0dec0d0d545a6e2fe54b84a6b519218727498d8c8dcc33f12d3eb49c68a35e", 357335, 1),
    "30188": ("5c350920ad0ebbb56939bfd471f73b187ef740611adcce384f6113af060c2612", 187616, 1),
    "30189": ("a80b799537199bf47b0a2772679f8601949c1194bf3a959308525ee903f901f7", 199932, 1),
    "30190": ("8b4d6735b8b0fe90501db8d53f1674545bc512b79d1e925ccb12c856dba3796d", 386333, 1),
    "30191": ("fa682651491003ffd246fb30a9ba9376fd78af5a6bdc961fc870cc12570a3377", 618329, 1),
    "30192": ("7f6ef4ec61f6cc8aaa65070cb5f4201d2484288386d681ffb8e0b84ee9ee32b3", 243450, 1),
    "30193": ("53d198c8df9643dcb7152c55f620f1b219bbbf7b0b4510c0639353e694a4c8ce", 44323, 1),
    "30194": ("039be67d22aa56c7bcba4e4c5c2f249c511dfdc3a823927be81ad4a4fe9f93c4", 273518, 1),
    "30195": ("60c5ebc12af5d1c47e449060ff2c96752615840259096e778daa19168c21abd5", 272460, 1),
    "30196": ("43236a3e3aa3536367eeb3e45ee519bf91c426fc8fc893f86f7711ee477ca693", 258699, 1),
    "30197": ("9d7a4f6bd144f9275921e1a64ad42d5b846055b9f920440e278663485325dc0f", 337266, 1),
    "30198": ("d9d3f4c20e254b84a5303cced1a052f8f9ff9cadc148286fc1a963efa0e997e6", 294572, 1),
    "30199": ("be085478ba626565d2ba99aeb453586c8959fa8b8822ae48bd63e8debc2114ea", 372552, 1),
    "30200": ("f1fbe18ed7f737d6c663ae4733ee37931cd609eb2e45d3e8db1adc1e76f0e3af", 37859, 1),
    "30201": ("a4991289cb7d4404d3bc6fe3b5a1925bcba92ba31ac78726b42f936410d9621f", 68981, 1),
    "30202": ("812905be54f7b7ab11989787cb1c114cfeba3665349e8758616cfad549a33026", 355919, 1),
    "30203": ("173c85cbae224ec8c7d06f51aea47db2d68f89910810152163be1e5f41463954", 440762, 1),
    "30204": ("f1bc8a703eec0a7836f9fabb32c2700d60da2eadc04bb6315eeb3705551da48d", 251996, 1),
    "30205": ("852db1e13187236fdff8ff1bd63f844971a4c368656bfbafcf8c1b41f2e5f03e", 38333, 1),
    "30206": ("1484e98cc751e96e892f26ffbcc4508140e58c98e7c7c933e1aaf532f9403667", 49518, 1),
    "30207": ("c9b6bd7af35250e7318d41428ddcf75a45e8b00002a5c054709d55548496371f", 278577, 1),
    "30208": ("d5896c05cbd6a1273673eb496feb18941dc5b44b560b348b63c8558ca827444b", 489301, 1),
    "30209": ("26757ee76e1b09fe04a9530e3b4be9be80345783de692d19d8e69614f9a673cc", 279173, 1),
    "30210": ("4e310dad94cbee910f0a277e15ea24de4ff92e7e55d24563e2e03ccd99d2dc2c", 295853, 1),
    "30211": ("4b4e5435268a6e325e5e8fad99c0b74e56d314fa235f1c47e84b4c48a5c99d82", 250323, 1),
    "30212": ("cd7e6147ab7e014c6d799f013fc6e7c53509b53d7d072828a8ebd43d5f6e3a34", 218259, 1),
    "30213": ("f26de9ec2acd40c2dc0d1f8814480b5f54997854c10410945251d16120faf387", 145243, 1),
    "30214": ("d50bab939bcfefa84a1b5b56653509eef46573d33dcccd0149a9a85a4bc99249", 357753, 1),
    "30215": ("8b7d43acf9c41084f388a7617fef58782522355fe92412f76b744378cda8a39f", 376005, 1),
    "30216": ("f883cfffd0cd55aee259045a09d8cef94bd8b4aeab80ef41fed0d59e1edb1636", 265692, 1),
    "30217": ("028009123812a9a4895a8fe58dec260a686217749ed74397dc5ada3be176d698", 321431, 1),
    "30218": ("7a260750f60518dd2de90972d6974e07c03fc4e4a20895f20b6623692bed2ea1", 376986, 1),
    "30219": ("1516f0e8182877113566c45acf10e5cbfe863f6b895621babd9defcaa3ac597b", 348792, 1),
    "30220": ("a3f9f11eb9aae1b4ab7a4d5942395c4476a42fc06b218ce36a568997dee08a59", 216677, 1),
    "30221": ("7ffbcdb625b8059f33fbfe56f92260e5e53aa31ed4a30e1c87225eb5ea543178", 199144, 1),
    "30222": ("75737a0bdd9f9e65a1803a5e00bc1ac01dba335ffb99de9103021ff2686074de", 289726, 1),
    "30223": ("8470bf75375a3664c655fc13236b98024dadad2c5d8301eed356a00274240064", 63359, 1),
    "30224": ("adbeecd8c94d4e62ada4547fd89b8ac6fb7125f5d34469b0649957fb77291b4a", 407401, 1),
    "30225": ("e56a1d04081999d8eb449248f2a210b643a6eccdfb0c88561973f96efc4fe1df", 420863, 1),
    "30226": ("738311daf02f7665002ef15bb4ca9d44dffd05d245e5ff0345b29b4a7ef65cf0", 441471, 1),
    "30227": ("8d4d6c4a68d87dc44890f4559076476c0ba924af52b9f780276129ebf61bd1ba", 92450, 1),
    "30228": ("46feb453c027cc960fd0744ba7070c47840cb423538ad9485bc68e4f46f22d52", 28599, 1),
    "30229": ("bbd6f52f9bfa0bd5e0905174e2419043d6828b46534cdbc13af03f88b3d897be", 138068, 1),
    "30230": ("24df119f7fad01595f059336ba03b83e92d2a4853eae7e1fa25e37443742a099", 185017, 1),
    "30231": ("3e6145b3aa79f134c65f5bcb0d63ce991d785046853fde7989d6a7a693ef1650", 498384, 1),
    "30232": ("938cf66361f49648a4cbb9f28fec5fbb38403e5b0565c726b07294d796798cee", 27220, 1),
    "30233": ("33dea589ddbc4eb1fd1ea672b0d8a5dbf9718c12253b1ec0de70673212aebaaa", 298512, 1),
    "30234": ("67d181603f2ec8e488807692b4d818acd665a2368ff2f7285afee14c8d3cabfe", 220889, 1),
    "30235": ("ee9d6a1421f1f09033820e81a515ab1966545b5694d7c87db5b95e36e09b7f93", 180550, 1),
    "30236": ("542a7d37e8438ee2b9eacbf6ec30237f15607ea5171d3d007e766dfddf7ebb40", 357536, 1),
    "30237": ("1cf1bdb965899183939b573dc2d84b80e53e3684dd3f02d82acfc1075e9b03ed", 367479, 1),
    "30238": ("be7104f9d4ed545d305ccfd13628bef77a1e8b14bead75f99802fca58866fb96", 58049, 1),
    "30239": ("5e365040794fbeff577b927393be7f3cd91141035f5958f91e3294406ce46ade", 386870, 1),
    "30240": ("1b33469ca2566f6559561dfddc2dd2cf10b29eab70757f06506aa1815c043273", 87274, 1),
    "30241": ("76ab39eee72793742977606312e5f6e1d1fcd3c069684bd9b47a10f9033120ab", 399199, 1),
    "30242": ("eb5a8a84c5f10f75f7b0d6c61c0aa221eb8f4952f861d58c539ed51dd1111f39", 260397, 1),
    "30243": ("63397cc3d9a4948393a8296cc4578eb992550d12fae1b7fffefab84ef054246d", 371299, 1),
    "30244": ("a1a29a6e21587c291787b7128404cd236bda8718228924fb5eee2a0c23a60d01", 252389, 1),
    "30245": ("57e9e77cda277756f13fa76f6f7a58c05f30cb8f2c96b4d96662dc66a64656ee", 75888, 1),
    "30246": ("2e731a344aa1d106c8b48329b9f04fb357178940d23b92b49b1e4a943ad79250", 53032, 1),
    "30247": ("b6939651c3f5f7bc1cae3ad34cda2ee03a9fdc4d1031b054962312809b723b3c", 141063, 1),
    "30248": ("7ff3d93c23a87d0e4784326d84de22d831c0d0b875f2032e283ccdce3256b18c", 457007, 1),
    "30249": ("231c59181321a807dfd5f8e5a42d126583bb986da4719013b4423704a74c203f", 138059, 1),
    "30250": ("77bb7ab9c66b04e279c75f12f66362111811a153f61bbeac2a5bc077be0feba7", 189878, 1),
    "30251": ("52e4ccff1f7d088fac23e366de9a6d223ddc0aa86d9c81bbb9d922d14bf34f4d", 151400, 1),
    "30252": ("807ab10469fc905e0d0c819adef5ce4e3ab62a8937c7bacbb764697bc726a7ca", 24031, 1),
    "30253": ("ff625e4fd58d04aa89b1c62f68fce3dc1a47ac7a07c5e99477486d1a7a681008", 243997, 1),
    "30254": ("080eae73d5ae9ce253f2cd4d961cabdb432f30be8a0692d5ed6588025b903da8", 329973, 1),
    "30255": ("67d49add48d49e86b3819648d5544889a2f03af03664e71fab08da3668956539", 287784, 1),
    "30256": ("fa75d8391fcf1cd259b3891e53dcd434f88072dd6c962a41298411b7289e7282", 358711, 1),
    "30257": ("98487449ff0c4cd622a97bc14bb35eafbd3a01d28f9ccfba23cd0f0e39a25279", 253247, 1),
    "30258": ("5a64eb02508a62391b106261c1a5a7531dd9a7de409b7d63cdfa49380a90586e", 223557, 1),
    "30259": ("72f209af323961a6cd3b7aa471c5b675f1460178b122908a4a56dc01a50797c9", 438599, 1),
    "30260": ("a027c62e8b1a412476931c39f5faea8cf4474f30683346757104b9012a62a3a6", 273880, 1),
    "30261": ("ab9a54b12da5580b9552caacebf597243b0840b094bf5af06aa94f955c580b7d", 171142, 1),
    "30262": ("d56e67ac6d792ab50790e013ce2d547374380ad1b9d110ca299b0233e5c1ddba", 227935, 1),
    "30263": ("e67a81e6d940057ebe82efa76b6780d1170ce699516486127717fb877c0aa637", 27678, 1),
    "30264": ("9fc039be115f3568bd428e022806899f993caaf734e70eb9cc245cd8a41b1c4d", 353784, 1),
    "30265": ("950e38a48609eeb2c3b7eb52134c0b2b9b3f7df3ee393cb62ad8cdcb8bcdcf30", 297373, 1),
    "30266": ("98436a81ffa1214e8491fd11e3d8af51ed5814ed347cdf261e74937f3ced04f3", 271224, 1),
    "30267": ("e1d1f0fa9ad7485a7be965fa922f459ee69511e2d2bf817139a68bcad7853435", 333280, 1),
    "30268": ("bd464d28a6a3b4edfddace93ebfd1e58860bbdc3560135680741a96dc2a69480", 326573, 1),
    "30269": ("c17fb7eb59086dba18ba58eb93bb34cfe0d6b99502ff867ebcc7b5cf17c2f3f2", 181867, 1),
    "30270": ("47223eb9e1f315fcaea373e3978bb4999416e9cf2714dec143ac4d66d74288a3", 34998, 1),
    "30271": ("503c78b99c86dbef5bbbb0ec7dda7fcfd2cfd9bdfc537234c1b983304b3ef2af", 299577, 1),
    "30272": ("5fd05dbe93ef552997c0d4a1022f02b8b08dd9fe9c91719b8dd669eaeb3950d8", 207775, 1),
    "30273": ("5095a2fafd904c37298eab32f0f0cb7605552ca1106d5d1365a02da687c1c9ea", 190454, 1),
    "30274": ("5a283b5a45f2e96ab1d716019e8f5a6424ee87fff0545e92e2625a3b2b45f558", 385055, 1),
    "30275": ("4801502dae7a3eae492cf63430f3547f28ea5d93548eeb61c6b01751c0df7789", 465091, 1),
    "30276": ("b5ea1778654ff9d8267e611fd8aa211e39a9889137e2dc3bac32173132684429", 269059, 1),
    "30277": ("b6cd404d7bea6c45d844cf9a2c62290c3605f85dcb6e831678870c454fdf1068", 184341, 1),
    "30278": ("64f9c046c11d2344034eb6c675fb0c1ab457adc310624dbe5f6683528fb75d84", 150503, 1),
    "30279": ("1f5b791067f9fbea5f45274177f34f4fa934c5c1eeb0b532cfcca03fa58b1036", 417110, 1),
    "30280": ("1c43d79e999fbb3425908acca0d867e6b054f8f72ff69ce741c6aae1441f8928", 326208, 1),
    "30281": ("f4d770912af0629c02ce19392aba7af8ff1680bdb7d277cf3574636d35a4f7b7", 223568, 1),
    "30282": ("83b5409cd74d6727f8e6fdddb556377e20f2f0248d62b00c3fff12efb0bcb977", 297120, 1),
    "30283": ("503abbff44f453af78564de96bf0817eb3217670283f1e4b576d7e4dd96848ab", 30026, 1),
    "30284": ("2c87ce1be62a12c13c7aab8460298a94ec8fcdcd82c22278e594b7fe6c7c9eeb", 430920, 1),
    "30285": ("9554f5ee3413793ffb1b0532686c0376486d53f5c3e67a9ad1e3d7a181cc2e10", 388442, 1),
    "30286": ("f8762bece142ed0583941ab9ec2f1d341353a0b2f06c780368dbbaf772b61e4f", 255025, 1),
    "30287": ("a4d782ad21b63ff13e95ff74b292b886551bdf674ebe6c24f5a098847b82f3bf", 299345, 1),
    "30288": ("906bfc30bb4f176f163552f865e3d6d80e4c74ba4080e0b32ebe23ff7c7d624f", 132493, 1),
    "30289": ("df9e5e07db827f3fdf8c0c983ebe22b7a66626614bb0807de7224d4b002569d3", 441063, 1),
    "30290": ("bc3d197da7cd913ce3fc185eda51f0a278563120a901a3be1101c4b3a0ede0c3", 396493, 1),
    "30291": ("8d9b803a55b9e6d82e41b8d0cf84866faa2eeba376911eadb826c1636c12cfa6", 304309, 1),
    "30292": ("2a53ac7b6d0890902b243923ec3c7c9bc2b2e92cbab1833238a168b23ba5d51f", 312133, 1),
    "30293": ("c5fd42f4be3ca8de7e22ca5df8824f7ed423b1febbbc6b84679e4fc849b6aad9", 40452, 1),
    "30294": ("e1060d2ebad1c48d2731814bb41daf430b05f843c2ec6734aaa5278cddbde21a", 205282, 1),
    "30295": ("b8e1e0d25166385d51ebef73a2ddfcd5606f1da12d87881480cb1555dbd0ef62", 383780, 1),
    "30296": ("246c24e8680d3e3bb6b12e5e817e27b15b80bfb21b74b93a3b8284f36b044c77", 29830, 1),
    "30297": ("7ae4bf7774bf9326b9cc7706c0565bb0d7cf641976aa1187a09e888953ab2901", 334764, 1),
    "30298": ("d1a431c5d20dded4cdb7d767f8fbc6697063c8440c330bdbbd99f0d30f062291", 390667, 1),
    "30299": ("12e7b5f142e4045476661d892032ddc20fbd9b081e007458117df6667dd24d35", 243801, 1),
    "30300": ("42f5fa6758fc3bd988ee8103d5e1add56703165959ba1f1e60dd92820df036aa", 53976, 1),
    "30301": ("ab4799afe1eae5b845a6fc63cca4b865a3a7b4e467204ae2dd6be5be8a7c13cd", 204845, 1),
    "30302": ("93fe2c943e866db04919b35934d50625cec17885ea37771b0a310be5fead8a5f", 20467, 1),
}


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def text_digest(rows: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 of the decoded text columns as a canonical JSON list of
    `[id, captions, question_type, text_detected]`."""
    payload = [
        [str(r["id"]), [str(c) for c in r["captions"]], str(r["question_type"]), bool(r["text_detected"])]
        for r in rows
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to
    read a parquet footer and a few column chunks without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _open_shard() -> Any:
    """The pinned shard as a `pyarrow.parquet.ParquetFile` over HTTPS range requests, after the file's
    declared size and LFS SHA-256 have been checked against the pins."""
    import pyarrow.parquet as pq
    from huggingface_hub import get_hf_file_metadata, hf_hub_url

    url = hf_hub_url(CORPUS_REPO, CORPUS_FILE["path"], repo_type="dataset", revision=CORPUS_REVISION)
    metadata = get_hf_file_metadata(url)
    declared = (metadata.etag or "").strip('"')
    if metadata.size != CORPUS_FILE["bytes"] or declared != CORPUS_FILE["sha256"]:
        raise ValueError(
            f"caption shard: the Hub declares {metadata.size} bytes / sha256 {declared[:16]}…, "
            f"pinned {CORPUS_FILE['bytes']} / {CORPUS_FILE['sha256'][:16]}…"
        )
    return pq.ParquetFile(_HttpRangeFile(url, CORPUS_FILE["bytes"]))


def _hub_text() -> list[dict[str, Any]]:
    rows = _open_shard().read(columns=list(CORPUS_TEXT_COLUMNS)).to_pylist()
    return [
        {
            "id": str(r["id"]),
            "captions": [str(c) for c in r["answer"]],
            "question_type": str(r["question_type"]),
            "text_detected": bool(r["text_detected"]),
        }
        for r in rows
    ]


def _hub_images(row_groups: Sequence[int] | None = None) -> dict[str, bytes]:
    """The `media` column of the pinned row groups only (one range read each): image id -> JPEG bytes."""
    shard = _open_shard()
    out: dict[str, bytes] = {}
    for group in row_groups if row_groups is not None else CORPUS_FILE["row_groups"]:
        if group not in CORPUS_FILE["row_groups"]:
            raise ValueError(f"row group {group} is not pinned")
        _read_group(shard, group, out)
    return out


def _read_group(shard: Any, group: int, out: dict[str, bytes]) -> None:
    table = shard.read_row_group(group, columns=["id", CORPUS_IMAGE_COLUMN])
    for row in table.to_pylist():
        media = row[CORPUS_IMAGE_COLUMN]
        if not isinstance(media, list) or len(media) != 1:
            raise ValueError(
                f"row {row['id']}: expected exactly one image, found {len(media) if media else 0}"
            )
        out[str(row["id"])] = bytes(media[0]["bytes"])


def fetch_annotations(
    *, cache_dir: str | Path | None = None, fetcher: Callable[[], Sequence[Mapping[str, Any]]] | None = None
) -> list[dict[str, Any]]:
    """Return the shard's 1,550 caption rows from the cache or the Hub, digest-verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / "annotations.json"
    rows: list[dict[str, Any]] | None = None
    if local.is_file():
        cached = json.loads(local.read_text(encoding="utf-8"))
        if isinstance(cached, list) and text_digest(cached) == CORPUS_FILE["text_sha256"]:
            rows = cached
    if rows is None:
        raw = fetcher() if fetcher is not None else _hub_text()
        rows = [
            {
                "id": str(r["id"]),
                "captions": [str(c) for c in r["captions"]],
                "question_type": str(r["question_type"]),
                "text_detected": bool(r["text_detected"]),
            }
            for r in raw
        ]
        if len(rows) != CORPUS_FILE["rows"] or text_digest(rows) != CORPUS_FILE["text_sha256"]:
            raise ValueError(
                f"caption shard: fetched {len(rows)} rows with text sha256 {text_digest(rows)[:16]}…, "
                f"pinned {CORPUS_FILE['rows']} / {CORPUS_FILE['text_sha256'][:16]}…"
            )
        local.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
    return rows


def fetch_images(
    image_ids: Sequence[str],
    *,
    cache_dir: str | Path | None = None,
    fetcher: Callable[[Sequence[int]], Mapping[str, bytes]] | None = None,
) -> dict[str, Path]:
    """Stage the pinned photographs into the cache: cached files are re-hashed; anything missing or drifted is
    read from its pinned row group (one range read of that row group's image column; the fetcher receives
    the sorted row groups that are needed) and refused on any size or SHA-256 mismatch. Returns image id ->
    path."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    (cache / "images").mkdir(parents=True, exist_ok=True)
    out: dict[str, Path] = {}
    missing: list[str] = []
    for image_id in image_ids:
        if image_id not in IMAGE_PINS:
            raise ValueError(f"{image_id} is not one of the {len(IMAGE_PINS)} pinned photographs")
        digest, size, _group = IMAGE_PINS[image_id]
        dest = cache / "images" / f"{image_id}.jpg"
        data = dest.read_bytes() if dest.is_file() else None
        if data is None or len(data) != size or _sha256_bytes(data) != digest:
            missing.append(image_id)
        out[image_id] = dest
    if missing:
        groups = sorted({IMAGE_PINS[image_id][2] for image_id in missing})
        payload = fetcher(groups) if fetcher is not None else _hub_images(groups)
        for image_id in missing:
            digest, size, _group = IMAGE_PINS[image_id]
            data = payload.get(image_id)
            if data is None or len(data) != size or _sha256_bytes(data) != digest:
                got = (
                    f"{len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…"
                    if data is not None
                    else "no bytes"
                )
                raise ValueError(f"{image_id}.jpg: read {got}, pinned {size} / {digest[:16]}…")
            (cache / "images" / f"{image_id}.jpg").write_bytes(data)
    return out


def build_sample_dataset(
    annotations: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
    image_paths: Mapping[str, Path] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The pinned row-group-0 photographs that carry at least one caption, shuffled with `seed` and cut **by
    image** into `sizes`; every captioned photograph of `GALLERY_ROW_GROUP` is then appended to the test
    split, so retrieval is scored over a gallery several times larger than the held-out core (391
    photographs by default). `image_paths` (from `fetch_images`) fills each record's `image`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    by_id = {str(r["id"]): r for r in annotations}
    missing = [image_id for image_id in IMAGE_PINS if image_id not in by_id]
    if missing:
        raise ValueError(
            f"{len(missing)} pinned photograph(s) are absent from the annotations: {missing[:3]}"
        )
    usable = {
        image_id: list(
            dict.fromkeys(
                " ".join(str(c).split())
                for c in by_id[image_id]["captions"]
                if len(" ".join(str(c).split())) <= MAX_CAPTION_CHARS
            )
        )
        for image_id in IMAGE_PINS
    }
    pool = sorted(
        image_id
        for image_id, captions in usable.items()
        if len(captions) >= MIN_CAPTIONS and IMAGE_PINS[image_id][2] == 0
    )
    extra = sorted(
        image_id
        for image_id, captions in usable.items()
        if len(captions) >= MIN_CAPTIONS and IMAGE_PINS[image_id][2] == GALLERY_ROW_GROUP
    )
    if sum(sizes.values()) > len(pool):
        raise ValueError(
            f"{len(pool)} pinned photographs carry a caption; the split sizes need {sum(sizes.values())}"
        )
    random.Random(seed).shuffle(pool)
    out: dict[str, list[dict[str, Any]]] = {}
    offset = 0
    for name in ("train", "validation", "test"):
        chosen = pool[offset : offset + sizes[name]]
        offset += sizes[name]
        if name == "test":
            chosen = chosen + extra
        out[name] = [
            {
                "id": f"{name}-{index:04d}",
                "image_id": image_id,
                "image": str(image_paths[image_id])
                if image_paths is not None and image_id in image_paths
                else f"{image_id}.jpg",
                "captions": usable[image_id],
                "category": "text" if by_id[image_id]["text_detected"] else "no-text",
            }
            for index, image_id in enumerate(chosen)
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    annotation_fetcher: Callable[[], Sequence[Mapping[str, Any]]] | None = None,
    image_fetcher: Callable[[], Mapping[str, bytes]] | None = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned annotations and photographs."""
    annotations = fetch_annotations(cache_dir=cache_dir, fetcher=annotation_fetcher)
    paths = fetch_images(sorted(IMAGE_PINS), cache_dir=cache_dir, fetcher=image_fetcher)
    return build_sample_dataset(annotations, seed=seed, sizes=sizes, image_paths=paths)


def _check_record(record: Any, index: int, *, base_dir: Path | None) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/image/captions")
    for key in ("id", "image", "captions"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    image_ref = record["image"]
    if not isinstance(image_ref, (str, Path)) or not str(image_ref).strip():
        raise ValueError(f"{label}: image must be a file path")
    path = Path(image_ref)
    if not path.is_absolute() and base_dir is not None:
        path = base_dir / path
    if not path.is_file():
        raise ValueError(f"{label}: image file not found: {path}")
    try:
        with Image.open(path) as handle:
            handle.load()
            validate_image(handle)
            width, height = handle.size
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: {exc}") from exc
    except OSError as exc:
        raise ValueError(f"{label}: image cannot be decoded: {exc}") from exc
    captions = record["captions"]
    if isinstance(captions, str) or not isinstance(captions, Sequence) or len(captions) < MIN_CAPTIONS:
        raise ValueError(f"{label}: captions must be a list of at least {MIN_CAPTIONS} reference caption(s)")
    checked_captions = [" ".join(str(c).split()) for c in captions]
    if not all(checked_captions):
        raise ValueError(f"{label}: every reference caption must be a non-empty string")
    if len(set(checked_captions)) != len(checked_captions):
        raise ValueError(f"{label}: reference captions must be distinct")
    if any(len(c) > MAX_CAPTION_CHARS for c in checked_captions):
        raise ValueError(f"{label}: a reference caption exceeds MAX_CAPTION_CHARS={MAX_CAPTION_CHARS}")
    return {
        "id": rid,
        "image_id": str(record.get("image_id", rid)),
        "image": str(path),
        "image_size": [width, height],
        "captions": checked_captions,
        "category": str(record.get("category", "other")),
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    base_dir: str | Path | None = None,
) -> dict[str, Any]:
    """Structural validation of a captioning dataset (every image opened and decoded); raises ValueError
    before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, image, captions} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    base = Path(base_dir) if base_dir is not None else None
    checked = []
    ids: set[str] = set()
    images: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index, base_dir=base)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        images.add(item["image_id"])
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_images": len(images),
        "categories": dict(Counter(r["category"] for r in checked)),
        "captions_per_image": {
            "min": min(len(r["captions"]) for r in checked),
            "max": max(len(r["captions"]) for r in checked),
        },
        "caption_words": {
            "min": min(len(c.split()) for r in checked for c in r["captions"]),
            "max": max(len(c.split()) for r in checked for c in r["captions"]),
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["image_id"], list(r["captions"]), r.get("category", "")] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def reference_captions(record: Mapping[str, Any]) -> list[str]:
    """The matching captions of a record."""
    return [str(c) for c in record["captions"]]


def query_caption(record: Mapping[str, Any]) -> str:
    """The record's retrieval query: its first matching caption."""
    return str(record["captions"][0])


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image id appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record.get("image_id", record["id"]))
            if key in seen and seen[key] != name:
                raise ValueError(f"image {key!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
    base_dir: str | Path | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded split of a BYOD dataset into train/validation/test **by image**: every record on the same image
    lands in the same split, so a test image is never seen in training."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, base_dir=base_dir)["records"]
    groups: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        groups.setdefault(record["image_id"], []).append(record)
    order = list(groups.values())
    random.Random(seed).shuffle(order)
    n_test = max(1, round(len(checked) * test_fraction))
    n_val = round(len(checked) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for group in order:
        if len(splits["test"]) < n_test:
            splits["test"].extend(group)
        elif len(splits["validation"]) < n_val:
            splits["validation"].extend(group)
        else:
            splits["train"].extend(group)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a JSON array or a JSONL file of ``{id, image, captions}`` objects; `image` paths are
    resolved relative to the file's directory by `validate_dataset(..., base_dir=...)`."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .json or .jsonl")


def write_dataset_jsonl(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One record per line in the shape `load_byod_dataset` reads back (image paths as given)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    keys = ("id", "image_id", "image", "captions", "category")
    with open(out, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps({k: record[k] for k in keys if k in record}, ensure_ascii=False) + "\n")
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `bed8ad38cb2d…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BlipItmPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "blip-itm-base-coco",
  "modelId": "Salesforce/blip-itm-base-coco",
  "revision": "bed8ad38cb2d04a5a4bdf2d071b3c3c0a4aa724c",
  "files": [
    {
      "path": "README.md",
      "bytes": 5492,
      "sha256": "db2e7ff1e647bc42d8b0d4cd3653ad65c1d24b5559a153e6e6ea2f3c12edd978"
    },
    {
      "path": "config.json",
      "bytes": 4560,
      "sha256": "3e6464c2ce7c54512ddb101c5e9a8e77f4c2d637be9e3d005667ccd4a34c6ef2"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 445,
      "sha256": "0aa66e2e9ac3ea3b5cd4388c35072e22db4e1cc1f96c7872bed07749c712ade1"
    },
    {
      "path": "pytorch_model.bin",
      "bytes": 895139697,
      "sha256": "017fb3e7f4e125f13a8a4717f1402dbe0d0bb877474b4a203db13a4447b0227f"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 125,
      "sha256": "b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3"
    },
    {
      "path": "tokenizer.json",
      "bytes": 711396,
      "sha256": "d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 456,
      "sha256": "86da6fdb761b02f73a05561aba71711c2d7c205fe1fd1744046173a410263925"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 896093679
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BlipItmPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. VizWiz photographs, captions and split

`fetch_annotations` reads the pinned shard's four text columns (or the cache under `weights/vizwiz-captions/`): it first checks the byte size and SHA-256 the Hub declares for the file against the pins, then reads only the parquet footer and those column chunks through `pyarrow` over HTTPS range requests, and refuses the decoded columns unless their SHA-256 matches. `fetch_images` stages the 672 pinned photographs of row groups 0 and 1 — each cached file is re-hashed; anything missing is read from its row group's image column in one range read and refused on any size or SHA-256 mismatch. `build_sample_dataset` keeps the photographs that carry at least one caption within the scoring ceiling (one caption of 1,391 in row group 0 is longer than 256 characters and is dropped; duplicates are removed), shuffles row group 0's 318 with `SPLIT_SEED` and cuts them **by image** into 208 / 40 / 70 training, validation and core-test records, then appends row group 1's 321 captioned photographs to the test split as the gallery. Each record is labelled `text` or `no-text` from the corpus's text-detected flag. `validate_dataset` then opens and decodes every image and checks every record against the contract, `check_split_disjoint` asserts no image is shared, and the training split is written to `outputs/blip_itm_train.jsonl` in the shape BYOD expects.

Look for: 1,550 annotation rows, 672 photographs, three digests, the category mix per split (a little over half the photographs contain text), captions per image between 1 and 5, a 391-record test split whose gallery holds 1,737 captions, and four refusal probes — a duplicate id, a missing image file, a duplicated caption and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import collections
import hashlib
import io
import json
import zipfile

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_dir = Path('work') / 'byod'
    byod_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            name = Path(member.filename).name
            if member.is_dir() or not name or name.startswith('.'):
                continue
            (byod_dir / name).write_bytes(archive.read(member))
    records_file = next(p for p in (byod_dir / 'records.jsonl', byod_dir / 'records.json') if p.is_file())
    records = load_byod_dataset(records_file)
    splits = split_dataset(records, seed=SPLIT_SEED, base_dir=byod_dir)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    annotations = fetch_annotations(cache_dir='weights/vizwiz-captions')
    image_paths = fetch_images(sorted(IMAGE_PINS), cache_dir='weights/vizwiz-captions')
    raw_rows = {'annotations': len(annotations), 'photographs': len(image_paths), 'row_groups': CORPUS_FILE['row_groups']}
    splits = build_sample_dataset(annotations, seed=SPLIT_SEED, image_paths=image_paths)
    data_source = f'{CORPUS_NAME} {CORPUS_RELEASE} ({CORPUS_LICENSE})'
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
categories = {name: manifest['categories'] for name, manifest in dataset_manifests.items()}
gallery_texts, gallery_owners = gallery(test_records)
write_dataset_jsonl(splits['train'], 'outputs/blip_itm_train.jsonl')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'gallery_captions': len(gallery_texts), 'text_sha256': CORPUS_FILE['text_sha256'][:16] + '...', 'pinned_photographs': len(IMAGE_PINS)})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_images': manifest['unique_images'], 'categories': manifest['categories'], 'captions_per_image': manifest['captions_per_image'], 'caption_words': manifest['caption_words'], 'digest': manifest['digest'][:16] + '...'}})
example = splits['train'][0]
print({'example': {'id': example['id'], 'image': Path(example['image']).name, 'size': example['image_size'], 'category': example['category'], 'query': query_caption(example), 'captions': len(example['captions'])}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in splits['train'][:8]],
    'missing image file': [{**splits['train'][0], 'image': 'work/does-not-exist.jpg'}, *splits['train'][1:8]],
    'duplicated caption': [{**splits['train'][0], 'captions': [splits['train'][0]['captions'][0]] * 2}, *splits['train'][1:8]],
    'too small': splits['train'][:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Score through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: three flat cartoon scenes — a red house with a tree, a beach with a red sailboat, an apple and an orange on a table — are drawn in code with Pillow (no text rendering, so their digests are stable across builds) with one authored caption each, a different image family from the photographs, and a grid the model will score again after adaptation. `validate_inputs` applies exactly the checks `score` applies (image sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE`, at most `MAX_IMAGES` × `MAX_TEXTS` distinct captions of up to `MAX_TEXT_CHARS`) and returns an input manifest; a duplicated caption is validated too and its rejection recorded as a finding. `score` returns the `itm_probability`, `itm_logit_match` and `cosine` grids plus per-image rankings. **Neither score is calibrated and neither abstains** — the ITM probability is a per-pair classifier output, the cosine is comparable only within a row or column, and a caption set with nothing that fits still produces a highest-scoring caption. As recorded in the model card, the repository's CPU smoke on this same grid put every image's own caption first by both scores (ITM 0.998 for the house, 0.634 for the beach, 0.278 for the fruit scene). The per-image `evaluation_report` on the 3×3 grid is `sample-sanity` — three authored pairs carry no dispersion; whether the model retrieves *well* is what Section 6 measures over 391 photographs.

In [ ]:
import time

import numpy as np
from PIL import Image, ImageDraw


def synthetic_scenes():
    """Three flat cartoon scenes drawn with Pillow (no text); returns [(name, image, authored caption)]."""
    house = Image.new('RGB', (640, 480), (135, 206, 235))  # sky
    d = ImageDraw.Draw(house)
    d.rectangle([0, 300, 640, 480], fill=(60, 179, 75))  # grass
    d.ellipse([500, 40, 600, 140], fill=(255, 215, 0))  # sun
    d.rectangle([120, 180, 320, 330], fill=(200, 40, 40))  # red house
    d.polygon([(100, 180), (220, 90), (340, 180)], fill=(90, 50, 20))  # brown roof
    d.rectangle([200, 260, 240, 330], fill=(70, 40, 20))  # brown door
    d.ellipse([420, 260, 520, 360], fill=(40, 100, 40))  # tree crown
    d.rectangle([460, 350, 480, 420], fill=(90, 60, 30))  # trunk
    d.ellipse([60, 380, 140, 440], fill=(255, 255, 255))  # white ball
    beach = Image.new('RGB', (640, 480), (120, 190, 240))  # sky
    d = ImageDraw.Draw(beach)
    d.rectangle([0, 220, 640, 330], fill=(30, 110, 200))  # sea
    d.rectangle([0, 330, 640, 480], fill=(238, 214, 150))  # sand
    d.ellipse([60, 40, 150, 130], fill=(255, 230, 80))  # sun
    d.polygon([(400, 330), (470, 330), (435, 210)], fill=(230, 40, 40))  # red sail
    d.rectangle([432, 210, 438, 330], fill=(90, 60, 30))  # mast
    d.ellipse([200, 370, 260, 430], fill=(255, 120, 40))  # beach ball
    fruit = Image.new('RGB', (480, 480), (250, 250, 245))
    d = ImageDraw.Draw(fruit)
    d.ellipse([60, 120, 220, 280], fill=(220, 30, 30))  # red apple
    d.rectangle([135, 95, 145, 125], fill=(80, 50, 20))  # stalk
    d.ellipse([250, 140, 430, 300], fill=(255, 170, 20))  # orange
    d.polygon([(90, 400), (400, 400), (360, 330), (130, 330)], fill=(180, 120, 60))  # table
    return [
        ('synthetic_house_640x480.png', house, 'a red house with a tree under a blue sky'),
        ('synthetic_beach_640x480.png', beach, 'a sailboat on the sea next to a beach'),
        ('synthetic_fruit_480x480.png', fruit, 'an apple and an orange on a wooden table'),
    ]


scenes = synthetic_scenes()
scene_names = [name for name, _, _ in scenes]
scene_images = [image for _, image, _ in scenes]
scene_texts = [caption for _, _, caption in scenes]
correct_text_per_image = list(range(len(scenes)))  # caption i describes image i
scene_digests = {name: hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest() for name, image in zip(scene_names, scene_images)}
ceilings = {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'IMAGE_SIZE': IMAGE_SIZE, 'MAX_IMAGES': MAX_IMAGES, 'MAX_TEXTS': MAX_TEXTS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'MIN_CAPTIONS': MIN_CAPTIONS, 'MAX_CAPTION_CHARS': MAX_CAPTION_CHARS}
print(ceilings)
input_manifest = validate_inputs(scene_images, scene_texts, names=scene_names)
try:
    validate_inputs(scene_images, [scene_texts[0], '  ' + scene_texts[0] + ' '])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'duplicate-caption-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/blip_itm_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'scenes': scene_names, 'rgb_sha256': {k: v[:16] + '...' for k, v in scene_digests.items()}, 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})
t0 = time.perf_counter()
result = pipe.score(scene_images, scene_texts)
np.set_printoptions(precision=3, suppress=True)
print({'seconds': round(time.perf_counter() - t0, 2), 'pairs': result['n_images'] * len(result['texts'])})
print('itm_probability [image][text]:')
print(result['itm_probability'])
print('cosine [image][text]:')
print(result['cosine'])
checks = {
    'grid_shape': result['itm_probability'].shape == (len(scenes), len(scene_texts)),
    'probabilities_in_range': bool(np.all((result['itm_probability'] >= 0) & (result['itm_probability'] <= 1))),
    'cosines_finite': bool(np.all(np.isfinite(result['cosine']))),
    'one_ranking_per_image': len(result['rankings']) == len(scenes),
}
if not all(checks.values()):
    raise RuntimeError(f'score output failed a sanity check: {checks}')
frozen_scene = evaluation_report(result, correct_text_per_image, sample_kind='synthetic')
print({'checks': checks, 'frozen_scene': {m['id']: round(m['value'], 3) for m in frozen_scene['metrics']}, 'verdict': frozen_scene['verdict']})

## 6. Baselines and the frozen model's retrieval over the gallery

Three systems frame the adaptation. **Chance** is the analytical expectation of a random ranking over the same gallery (recall@k ≈ k / 391 in the text-to-image direction). The **colour-keyword nearest neighbour** retrieves with no model: text → image through the query's closest training caption by bag-of-words F1 and that photograph's 3×3 mean-colour grid; image → text through the colour-nearest training photograph's captions. The **frozen model** is scored by `pipe.evaluate`: every caption of every test record is the gallery (1,737 captions for 391 photographs), the ITC cosine grid gives **recall@1/5/10** in both directions (image → text counts a hit when any of the photograph's own captions is in the top k), **median rank** and **rsum** (the sum of the six recalls, 0..6); then the ITM head re-ranks each photograph's top-5 captions and each query caption's top-5 photographs (`itm_i2t_recall_at_1`, `itm_t2i_recall_at_1`) and is scored on **ITM pair accuracy** — each photograph's query caption against the hardest wrong caption by ITC. Expect the frozen model far above both baselines — it is a trained retriever — and read the per-category breakdown: the build record measured i2t R@1 0.742 / t2i R@1 0.652 / rsum 5.02 frozen, with the `text` photographs (labels, screens, packaging: i2t R@1 0.73, t2i 0.63) harder than the `no-text` ones (0.81 / 0.78).

In [ ]:
RERANK_TOP_K = 5  # @param {type:"integer"}

baseline_chance = chance_baseline(test_records)
baseline_neighbour = colour_keyword_baseline(train_records, test_records)
METRICS = ('i2t_recall_at_1', 'i2t_recall_at_5', 'i2t_recall_at_10', 't2i_recall_at_1', 't2i_recall_at_5', 't2i_recall_at_10', 'rsum')
ITM_METRICS = ('itm_i2t_recall_at_1', 'itm_t2i_recall_at_1', 'itm_pair_accuracy')
print({'chance_baseline': {k: round(baseline_chance[k], 3) for k in METRICS}, 'n': baseline_chance['n_images'], 'note': baseline_chance['baseline']})
print({'colour_keyword_baseline': {k: round(baseline_neighbour[k], 3) for k in METRICS}, 'note': baseline_neighbour['baseline']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, rerank_top_k=RERANK_TOP_K)
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'itm': {k: round(frozen_test[k], 3) for k in ITM_METRICS}, 'median_rank': [frozen_test['i2t_median_rank'], frozen_test['t2i_median_rank']], 'n': [frozen_test['n_images'], frozen_test['n_captions']], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': METRIC_DEFINITIONS})
frozen_fields = frozen_test['by_category']  # each category scored as its own sub-gallery of the same grid
print({'by_category_frozen': frozen_fields})
assert frozen_test['rsum'] > baseline_chance['rsum'] and frozen_test['rsum'] > baseline_neighbour['rsum']

## 7. Bounded fine-tuning of the text encoder's last blocks, the projections and the ITM head

`pipe.adapt` trains only the last `TRAINABLE_TEXT_LAYERS` blocks of the fused text encoder (self-attention, cross-attention to the image and feed-forward), the two ITC projections and the ITM head — two blocks by default, 19,298,818 of 223,744,258 parameters; the vision encoder and the text embeddings stay frozen. The frozen vision encoder's output is computed **once** per training photograph and reused across epochs. Every (photograph, caption) pair is one sample, 888 per epoch here, and a batch of 16 trains BLIP's two objectives: the **image-text contrastive loss** (symmetric cross-entropy over the in-batch cosine similarities at temperature 0.07, pairs of the same photograph counted as positives) and the **image-text matching loss** (the ITM head on the batch's positives plus one hard negative caption per photograph and one hard negative photograph per caption, sampled in proportion to their ITC similarity, never from the same photograph). AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Epoch 0 records the frozen model's validation retrieval; every epoch is scored on the 40 validation photographs by ITC rsum, and the epoch with the highest validation rsum is kept — a 40-photograph gallery saturates near 5.9 of 6, so that selection is weakly informed, which is why the 391-photograph gallery in Section 8 is what the numbers are read from.

Watch the training loss fall from about 0.62 while validation rsum moves by hundredths: the frozen model already retrieves well, and what adapts is the alignment of VizWiz's caption style with its photographs. The build record's counter-examples — four blocks at 5e-5 for six epochs gained no more than two blocks at 2e-5 for four — are in the model card.

In [ ]:
EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 2e-5  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}
TRAINABLE_TEXT_LAYERS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in ('i2t_recall_at_1', 't2i_recall_at_1', 'rsum')})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_text_layers=TRAINABLE_TEXT_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'training_pairs': adapt_result['n_pairs'], 'itc_temperature': adapt_result['itc_temperature'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation over the gallery

The test photographs were never used for training or epoch selection, and no test image appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, the four systems are put side by side on the six recalls and rsum, the ITM re-ranking and pair accuracy are repeated, and the per-category recall@1 is repeated. Read it in this order: **rsum** first (the number the epoch was selected on), then **image → text R@1** (where the build record measured the gain: 0.742 → 0.791, and 0.803 → 0.844 after ITM re-ranking — carried by the `text` photographs, 0.73 → 0.80, while the `no-text` ones stay at 0.81) and **text → image R@1** (0.652 → 0.673), then the ITM pair accuracy, which barely moved in the build record (0.683 → 0.688) — the ITM head was already the better judge of hard pairs, and 208 photographs from one seeded split of one corpus gives **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on VizWiz says nothing about your photographs until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records, rerank_top_k=RERANK_TOP_K)
adapted_val = pipe.evaluate(val_records)
adapted_fields = adapted_test['by_category']
comparison = {metric: {'chance': round(baseline_chance[metric], 3), 'neighbour': round(baseline_neighbour[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['itm'] = {metric: {'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in ITM_METRICS}
comparison['median_rank'] = {'frozen': [frozen_test['i2t_median_rank'], frozen_test['t2i_median_rank']], 'adapted': [adapted_test['i2t_median_rank'], adapted_test['t2i_median_rank']]}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS + ITM_METRICS}
comparison['by_category'] = {category: {'n': frozen_fields[category]['n'], 'frozen': [round(frozen_fields[category]['i2t_recall_at_1'], 3), round(frozen_fields[category]['t2i_recall_at_1'], 3)], 'adapted': [round(adapted_fields[category]['i2t_recall_at_1'], 3), round(adapted_fields[category]['t2i_recall_at_1'], 3)]} for category in frozen_fields}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'categories': categories,
    'gallery': {'photographs': len(test_records), 'captions': len(gallery_texts)},
    'rerank_top_k': RERANK_TOP_K,
    'baselines': {'chance': baseline_chance, 'colour_keyword': baseline_neighbour},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/blip_itm_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['rsum'] > frozen_test['rsum']
print({'report': 'outputs/blip_itm_evaluation_report.json'})

## 9. Re-score the drawn scenes, export the adapter and reload it

The 3×3 grid from Section 5 is scored again by the adapted model — drawings, a different image family from the photographs it was tuned on, so this is a small look at what the adaptation did *outside* its corpus (the build record's grids are in the model card; a changed ranking here is a finding to record, not a failure) — and reported with the per-image `evaluation_report` (`sample-sanity`). Both grids are written as CSV.

`pipe.save_artifact` writes the trained tensors — the text encoder's last two blocks, the two projections and the ITM head, about 77 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `pytorch_model.bin`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `BlipItmPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the text encoder blocks, the projections and the ITM head, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical cosine grids on eight test photographs against their query captions (VER4).

In [ ]:
import csv
import shutil

adapted_result = pipe.score(scene_images, scene_texts)
adapted_scene = evaluation_report(adapted_result, correct_text_per_image, sample_kind='synthetic')
print('adapted itm_probability [image][text]:')
print(adapted_result['itm_probability'])
print({'scene_after_adaptation': {m['id']: round(m['value'], 3) for m in adapted_scene['metrics']}, 'verdict': adapted_scene['verdict']})
with open('outputs/blip_itm_scores.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'text', 'frozen_itm_probability', 'frozen_cosine', 'adapted_itm_probability', 'adapted_cosine'])
    for i, name in enumerate(scene_names):
        for j, text in enumerate(scene_texts):
            writer.writerow([name, text, f"{result['itm_probability'][i, j]:.4f}", f"{result['cosine'][i, j]:.4f}", f"{adapted_result['itm_probability'][i, j]:.4f}", f"{adapted_result['cosine'][i, j]:.4f}"])

artifact_dir = Path('outputs/blip_itm_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'blip_itm', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = BlipItmPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
parity_images = []
for record in test_records[:8]:
    with Image.open(record['image']) as photo:
        photo.load()
        parity_images.append(photo.copy())
parity_texts = [query_caption(r) for r in test_records[:8]]
before = pipe.score(parity_images, parity_texts)['cosine'].round(4)
after = reloaded.score(parity_images, parity_texts)['cosine'].round(4)
parity = {'identical_scores': int((before == after).sum()), 'of': int(before.size)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_scores'] == parity['of']

weight_entry = next(entry for entry in MANIFEST['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'pytorch_model.bin pickle, digest-verified, weights_only=True', 'weight_sha256': weight_entry['sha256'], 'hosted_tf_weight_file_not_loaded': HOSTED_TF_WEIGHT_FILE},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'text_columns': list(CORPUS_TEXT_COLUMNS), 'file': CORPUS_FILE, 'pinned_images': len(IMAGE_PINS), 'gallery_row_group': GALLERY_ROW_GROUP},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'scenes': {'names': scene_names, 'sizes': [list(image.size) for image in scene_images], 'rgb_sha256': scene_digests, 'texts': scene_texts, 'correct_text_per_image': correct_text_per_image}, 'frozen_grid': {'itm_probability': result['itm_probability'].tolist(), 'cosine': result['cosine'].tolist()}, 'adapted_grid': {'itm_probability': adapted_result['itm_probability'].tolist(), 'cosine': adapted_result['cosine'].tolist()}, 'frozen_report': frozen_scene, 'adapted_report': adapted_scene},
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/blip_itm_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen model is already a competent retriever on photographs it never saw — far above the two non-neural baselines, and at the ceiling of a 70-photograph gallery — and a bounded fine-tuning of the text encoder's last two blocks, the ITC projections and the ITM head on 208 VizWiz photographs moves recall over a 391-photograph gallery by a few points (rsum 5.02 → 5.16 in the build record: image → text R@1 0.742 → 0.791, text → image R@1 0.652 → 0.673, ITM-reranked image → text 0.803 → 0.844) with a 77 MB adapter that reloads to identical scores. That is the claim: the adaptation contract works end to end on a real out-of-distribution retrieval corpus, and the numbers it produces are read on six recalls, the ITM re-ranking and per category against two non-neural baselines and the frozen model rather than in isolation. The gallery size matters more than the adaptation does — the same frozen model scores rsum 5.72 on 70 photographs and 5.02 on 391 — and the notebook says so.

The gallery is 391 photographs from two row groups of one shard of one corpus, the validation split that picks the epoch is 40 photographs whose rsum saturates, the metrics are recall@k over a gallery (own implementations of the COCO retrieval protocol; none a human judgement), and the ITM pair accuracy did not move. So a gain here says the contract works, not that the adapted model is better on your photographs, that it reads the labels VizWiz captions transcribe, or that a match probability is trustworthy — it still ranks something first for every query, and it can be wrong confidently. Fine-tuning on a narrow corpus can also erode the model elsewhere; the drawn scenes re-scored in Section 9 are one 3×3 grid of evidence about that, not a measurement.

Three things to carry to real data. **Gallery first:** recall@k is only meaningful against a gallery of the deployment's own size — measure the frozen model over your gallery before any adapted number. **Leakage:** keep every record on an image in one split (the contract does this) and split by photographer or session when your images come from few sources, never at random over near-duplicate frames. **Negatives:** the in-batch negatives are what teach the ITM head; a corpus of near-identical photographs gives it hard negatives, a corpus of unrelated ones gives it easy ones and little to learn.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify the captions and photographs of a real captioning corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against two trivial baselines and the frozen model on an image-disjoint gallery, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, retrieval quality on any other population or gallery, or production fitness.

**Optional experiments (they do not affect the default path):** set `RERANK_TOP_K = 10` and compare the re-ranked recalls with the cost; set `TRAINABLE_TEXT_LAYERS = 4` and `LEARNING_RATE = 5e-5` and read the build record's counter-example against your own run; raise `EPOCHS` and watch the saturated validation rsum pick the epoch; or bring your own photographs through BYOD and read the chance baseline over *your* gallery before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/blip-itm-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/blip-itm-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/blip-itm-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model (Salesforce, BSD-3-Clause): https://huggingface.co/Salesforce/blip-itm-base-coco
- Upstream code: https://github.com/salesforce/BLIP
- BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation (Li et al., 2022): https://arxiv.org/abs/2201.12086
- Captioning Images Taken by People Who Are Blind (Gurari et al., ECCV 2020; VizWiz-Captions, CC BY 4.0): https://arxiv.org/abs/2002.08565 — data: https://vizwiz.org/tasks-and-datasets/image-captioning/
- Align before Fuse: Vision and Language Representation Learning with Momentum Distillation — the ITC + ITM objectives with hard-negative mining (Li et al., 2021): https://arxiv.org/abs/2107.07651
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)